# Load in and clean data

## Loading in Packages and Data

In [0]:
%pip install "matplotlib==3.8.4" matplotlib-venn
%pip install upsetplot
dbutils.library.restartPython()

In [0]:
import pandas as pd

results_GPT_full_df = pd.read_json(("/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full_final.json"), orient="records")
display(results_GPT_full_df)
print(len(results_GPT_full_df))

results_mini_full_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_GPT_Mini_Output_Results_full_final.json", orient="records")
display(results_mini_full_df)
print(len(results_mini_full_df))

results_claude_full_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_final.json", orient="records")
display(results_claude_full_df)
print(len(results_claude_full_df))

manual_score = pd.read_csv("/Workspace/Users/user/user/manual_score.csv")
display(manual_score)

notes_enrolled4 = pd.read_csv("/Workspace/Users/user/user/notes_enrolled4.csv")
display(notes_enrolled4)
print(len(notes_enrolled4))

checklist_filtered = pd.read_csv("/Workspace/Users/user/user/checklist_filtered.csv")
display(checklist_filtered)

## Data Cleaning

In [0]:
# list of notes that are in notes_enrolled4 but were not in the manual score. These notes are kept in the analysis. 
gpt_not_manual_ids = set(results_GPT_full_df['Subject ID']) - set(manual_score['Subject ID'])
display(notes_enrolled4[notes_enrolled4['Subject ID'].isin(gpt_not_manual_ids)])
len(gpt_not_manual_ids)

In [0]:
# Printing details about those in notes_enrolled4 but not in manual_score as well as in GPT results but not in manual_score

in_notes_not_manual = set(notes_enrolled4['Subject ID']) - set(manual_score['Subject ID'])
in_gpt_not_manual = set(results_GPT_full_df['Subject ID']) - set(manual_score['Subject ID'])

print(f"In notes_enrolled4 but not manual_score: {len(in_notes_not_manual)}")
print(f"In GPT results but not manual_score: {len(in_gpt_not_manual)}")
print(f"GPT-not-manual is subset of notes-not-manual: {in_gpt_not_manual.issubset(in_notes_not_manual)}")
print(in_notes_not_manual)

In [0]:
#Loading in redcap mapping
REDCap_mapping_full_path = "/Workspace/Users/user/user/data_labels_filtered4.csv"

REDCap_mapping = pd.read_csv(REDCap_mapping_full_path)
print('Number of rows:', len(REDCap_mapping))
print('Number of columns:', len(REDCap_mapping.columns))
display(REDCap_mapping)

In [0]:
# Check if the notes missing in the manual review appear in REDCap_mapping or in checklist_filtered
missing_in_redcap = set(in_notes_not_manual) - set(REDCap_mapping.values.flatten())
missing_in_checklist = set(in_notes_not_manual) - missing_in_redcap

print(f"Not in REDCap_mapping at all: {len(missing_in_redcap)}")
print(missing_in_redcap)
print(f"\nIn REDCap_mapping but phenotypes didn't match checklist_filtered: {len(missing_in_checklist)}")
print(missing_in_checklist)

In [0]:
# For those in REDCap_mapping, what phenotypes did they have?
# Recreate records from REDCap_mapping
records = []
for i in range(0, len(REDCap_mapping)):
    phen = []
    for j in range(0, len(REDCap_mapping.columns)):
        if pd.isnull(REDCap_mapping.iloc[i][j]) == False:
            phen.append(REDCap_mapping.iloc[i][j])
    records.append(phen)

# Now check phenotypes for the 11 missing Subject IDs
for sid in missing_in_checklist:
    sid_row = [i for i, record in enumerate(records) if record and record[0] == sid]
    if sid_row:
        phenotypes = [x for x in records[sid_row[0]] if not str(x).startswith('WES')]
        print(f"\n{sid}: {phenotypes}")
    else:
        print(f"\n{sid}: not found in records")

In [0]:
print(f"notes_enrolled4 subjects: {notes_enrolled4['Subject ID'].nunique()}")
print(f"results_GPT_full_df subjects: {results_GPT_full_df['Subject ID'].nunique()}")
print(f"results_mini_full_df subjects: {results_mini_full_df['Subject ID'].nunique()}")
print(f"results_claude_full_df subjects: {results_claude_full_df['Subject ID'].nunique()}")
print(f"manual_score subjects: {manual_score['Subject ID'].nunique()}")

In [0]:
# Finding phenotypes that are not represented in gpt, claude, mini, and manual

phenotypes_results_gpt = set(results_GPT_full_df['Phenotype'].dropna().unique())
phenotypes_results_mini = set(results_mini_full_df['Phenotype'].dropna().unique())
phenotypes_results_claude = set(results_claude_full_df['Phenotype'].dropna().unique())

phenotypes_manual = set(manual_score['Phenotype'].dropna().unique())

missing_from_manual = sorted(phenotypes_results_gpt - phenotypes_manual)
missing_from_gpt = sorted(phenotypes_manual - phenotypes_results_gpt)
missing_from_mini = sorted(phenotypes_manual - phenotypes_results_mini)
missing_from_claude = sorted(phenotypes_manual - phenotypes_results_claude)

print(len(missing_from_manual), "values in GPT but not in manual_score")
print(missing_from_manual)

print()

print(len(missing_from_gpt), "values in manual_score but not in GPT")
print(missing_from_gpt)

print()

print(len(missing_from_mini), "values in manual_score but not in mini")
print(missing_from_mini)

print()

print(len(missing_from_claude), "values in manual_score but not in claude")
print(missing_from_claude)

In [0]:
# Remapping phenotypes such that any parenthesis/spacing issues are normalized and Chairi I malformation (which claude hallucinated and is a subset of the original phenotype that was input into the model) is renamed to chiari malformation, the original name

remap = {
    # Parenthesis fixes — all should have closing )
    'Double Outlet Right Ventricle (DORV': 'Double Outlet Right Ventricle (DORV)',
    'Hypertrophic cardiomyopathy (HCM': 'Hypertrophic cardiomyopathy (HCM)',
    'Tetralogy of Fallot (TOF': 'Tetralogy of Fallot (TOF)',
    'CPEO (opthalmoplegia': 'CPEO (opthalmoplegia)',
    'Autism Spectrum Disorder (ASD': 'Autism Spectrum Disorder (ASD)',
    'Diabetes Mellitus Type I (insulin dependent': 'Diabetes Mellitus Type I (insulin dependent)',
    'Diabetes Mellitus Type II (non-insulin dependent': 'Diabetes Mellitus Type II (non-insulin dependent)',
    # Hallucination fix
    'Chiari I malformation': 'Chiari malformation',
}

manual_score['Phenotype'] = manual_score['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
results_GPT_full_df['Phenotype'] = results_GPT_full_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
results_mini_full_df['Phenotype'] = results_mini_full_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
results_claude_full_df['Phenotype'] = results_claude_full_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
checklist_filtered['Term from checklist'] = checklist_filtered['Term from checklist'].str.strip().map(lambda x: remap.get(x, x))

In [0]:
# Showing how many duplicated phenotypes remain
results_GPT_full_df.duplicated(subset=['Subject ID', 'Phenotype']).sum()

In [0]:
# Finding phenotypes that are not represented in gpt, claude, mini, and manual after deduplication

phenotypes_results_gpt = set(results_GPT_full_df['Phenotype'].dropna().unique())
phenotypes_results_mini = set(results_mini_full_df['Phenotype'].dropna().unique())
phenotypes_results_claude = set(results_claude_full_df['Phenotype'].dropna().unique())

phenotypes_manual = set(manual_score['Phenotype'].dropna().unique())

missing_from_manual = sorted(phenotypes_results_gpt - phenotypes_manual)
missing_from_gpt = sorted(phenotypes_manual - phenotypes_results_gpt)
missing_from_mini = sorted(phenotypes_manual - phenotypes_results_mini)
missing_from_claude = sorted(phenotypes_manual - phenotypes_results_claude)

print(len(missing_from_manual), "values in results_GPT_full_df but not in manual_score")
print(missing_from_manual)

print()

print(len(missing_from_gpt), "values in manual_score but not in GPT")
print(missing_from_gpt)

print()

print(len(missing_from_mini), "values in manual_score but not in mini")
print(missing_from_mini)

print()

print(len(missing_from_claude), "values in manual_score but not in claude")
print(missing_from_claude)

In [0]:
#Renaming columns
results_GPT_full_df = results_GPT_full_df.rename(columns={'Score': 'GPT_Score'})
results_mini_full_df = results_mini_full_df.rename(columns={'Score': 'GPT_Mini_Score'})
results_claude_full_df = results_claude_full_df.rename(columns={'Score': 'Claude_Sonnet_Score'})

In [0]:
# Viewing score totals across entire dataframes (no subsetting)

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    if agg == 'count':
        score_totals[name] = df[score_col].count()
    else:
        score_totals[name] = df[score_col].sum()

print("Total phenotypes scored positive, per model, across all subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
# Displaying notes under 150 characters long 
short_notes = notes_enrolled4[notes_enrolled4['NOTE_TEXT'].str.len() < 150]
display(short_notes)

In [0]:
manual_score.columns

In [0]:
# Viewing score totals for rows to be dropped in ids_to_drop

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    subset = df[df['Subject ID'].isin(short_notes['Subject ID'])]
    if agg == 'count':
        score_totals[name] = subset[score_col].count()
    else:
        score_totals[name] = subset[score_col].sum()

print(f"Subjects with notes under 150 characters: {short_notes['Subject ID'].nunique()}")
print("\nTotal phenotypes scored positive, per model, for these subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
# Displayign the phenotypes that the manual had scored to these short notes
display(manual_score[manual_score['Subject ID'].isin(short_notes['Subject ID'])])

In [0]:
# Dropping notes under 150 characters long
ids_to_drop = short_notes['Subject ID']

manual_score = manual_score[~manual_score['Subject ID'].isin(ids_to_drop)]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(ids_to_drop)]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin(ids_to_drop)]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin(ids_to_drop)]

In [0]:
# Subjects where GPT scored 0 phenotypes total
gpt_totals = results_GPT_full_df.groupby('Subject ID')['GPT_Score'].sum()
zero_score_subjects = gpt_totals[gpt_totals == 0].index.tolist()

print(f"Subjects with 0 phenotypes scored by GPT: {len(zero_score_subjects)}")
print(zero_score_subjects)

# Pull their notes from notes_enrolled4
zero_score_notes = notes_enrolled4[notes_enrolled4['Subject ID'].isin(zero_score_subjects)]

print(f"\nMatching note rows found: {len(zero_score_notes)}")
display(zero_score_notes[['Subject ID', 'NOTE_TEXT', 'NOTE_TYPE']])

In [0]:
# Viewing score totals for rows dropped in next cell
subject_ids = [8 WES IDs]

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    subset = df[df['Subject ID'].isin(subject_ids)]
    if agg == 'count':
        score_totals[name] = subset[score_col].count()
    else:
        score_totals[name] = subset[score_col].sum()

print(f"Subjects to be dropped: {len(subject_ids)}")
print("\nTotal phenotypes scored positive, per model, for these subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
manual_score = manual_score[~manual_score['Subject ID'].isin([9 WES IDs])]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin([9 WES IDs])]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin([9 WES IDs])]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin([9 WES IDs])]

In [0]:
# Subjects where mini scored 0 phenotypes total
mini_totals = results_mini_full_df.groupby('Subject ID')['GPT_Mini_Score'].sum()
zero_score_subjects = mini_totals[mini_totals == 0].index.tolist()

print(f"Subjects with 0 phenotypes scored by mini: {len(zero_score_subjects)}")
print(zero_score_subjects)

# Pull their notes from notes_enrolled4
zero_score_notes = notes_enrolled4[notes_enrolled4['Subject ID'].isin(zero_score_subjects)]

print(f"\nMatching note rows found: {len(zero_score_notes)}")
display(zero_score_notes[['Subject ID', 'NOTE_TEXT']])

In [0]:
# Subjects where claude scored 0 phenotypes total
claude_totals = results_claude_full_df.groupby('Subject ID')['Claude_Sonnet_Score'].sum()
zero_score_subjects = claude_totals[claude_totals == 0].index.tolist()

print(f"Subjects with 0 phenotypes scored by claude: {len(zero_score_subjects)}")
print(zero_score_subjects)

# Pull their notes from notes_enrolled4
zero_score_notes = notes_enrolled4[notes_enrolled4['Subject ID'].isin(zero_score_subjects)]

print(f"\nMatching note rows found: {len(zero_score_notes)}")
display(zero_score_notes[['Subject ID', 'NOTE_TEXT']])

In [0]:
display(notes_enrolled4[notes_enrolled4['Subject ID'] == 1 WES ID])
display(manual_score[manual_score['Subject ID'] == 1 WES ID])
display(results_GPT_full_df[results_GPT_full_df['Subject ID'] == 1 WES ID])
display(results_mini_full_df[results_mini_full_df['Subject ID'] == 1 WES ID])
display(results_claude_full_df[results_claude_full_df['Subject ID'] == 1 WES ID])

In [0]:
matches = notes_enrolled4[notes_enrolled4['NOTE_TEXT'].str.contains('consent forms', case=True, na=False)]

print(f"Rows matching 'consent forms' (case-sensitive): {len(matches)}")
display(matches)

In [0]:
# Check that results_GPT_full_df, results_mini_full_df, and results_claude_full_df
# all have the exact same set of Subject IDs (no extras, no missing), and that
# all of those Subject IDs are also present in manual_score.

gpt_ids = set(results_GPT_full_df['Subject ID'].unique())
mini_ids = set(results_mini_full_df['Subject ID'].unique())
claude_ids = set(results_claude_full_df['Subject ID'].unique())
manual_ids = set(manual_score['Subject ID'].unique())

print(f"GPT subjects: {len(gpt_ids)}")
print(f"Mini subjects: {len(mini_ids)}")
print(f"Claude subjects: {len(claude_ids)}")
print(f"Manual subjects: {len(manual_ids)}")
print()

# --- Pairwise comparison across the three result sets ---
all_three_equal = (gpt_ids == mini_ids == claude_ids)
print(f"GPT, Mini, and Claude all have identical Subject ID sets: {all_three_equal}")
print()

if not all_three_equal:
    print("GPT vs Mini:")
    print(f"  In GPT but not Mini: {sorted(gpt_ids - mini_ids)}")
    print(f"  In Mini but not GPT: {sorted(mini_ids - gpt_ids)}")
    print()
    print("GPT vs Claude:")
    print(f"  In GPT but not Claude: {sorted(gpt_ids - claude_ids)}")
    print(f"  In Claude but not GPT: {sorted(claude_ids - gpt_ids)}")
    print()
    print("Mini vs Claude:")
    print(f"  In Mini but not Claude: {sorted(mini_ids - claude_ids)}")
    print(f"  In Claude but not Mini: {sorted(claude_ids - mini_ids)}")
    print()

# --- Check all three are fully contained in manual_score ---
combined_result_ids = gpt_ids | mini_ids | claude_ids
missing_from_manual = combined_result_ids - manual_ids

print(f"All GPT/Mini/Claude subjects present in manual_score: {len(missing_from_manual) == 0}")
if missing_from_manual:
    print(f"  Subjects missing from manual_score: {sorted(missing_from_manual)}")

# --- Optional: also flag if manual_score has extra subjects not in any result set ---
extra_in_manual = manual_ids - combined_result_ids
if extra_in_manual:
    print(f"\nSubjects in manual_score but not in any of GPT/Mini/Claude: {len(extra_in_manual)}")
    print(f"  {sorted(extra_in_manual)}")

In [0]:
notes_enrolled4.columns

In [0]:
# View the notes for GPT/Mini/Claude subjects not present in manual_score

subject_ids_to_check = combined_result_ids - manual_ids

notes_subset = notes_enrolled4[
    notes_enrolled4['Subject ID'].isin(subject_ids_to_check)
][['Subject ID', 'NOTE_FILE_TIME_DTTM', 'NOTE_TEXT', 'Enrollment Status', 'Date of last chart review:']].dropna(subset=['NOTE_TEXT']).reset_index(drop=True)

display(notes_subset)

In [0]:
import pandas as pd

output_folder = "/Workspace/Users/user/user/"

data_labels_filtered4 = pd.read_csv(output_folder + "data_labels_filtered4.csv")

matching_ids = set(data_labels_filtered4['Record ID #:']).intersection(set(subject_ids_to_check))

print(f"Total rows in data_labels_filtered4: {len(data_labels_filtered4)}")
print(f"Subject IDs in subject_ids_to_check: {len(subject_ids_to_check)}")
print(f"Matching subject IDs found: {len(matching_ids)}")

if matching_ids:
    print(matching_ids)
    display(data_labels_filtered4[data_labels_filtered4['Record ID #:'].isin(matching_ids)])
else:
    print("No matching subject IDs found.")

In [0]:
# It looks like these were evaluated, just not scored a phenotype by the manual scorer

# manual_score = manual_score[~manual_score['Subject ID'].isin(subject_ids_to_check)]
#results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(subject_ids_to_check)]
#results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin(subject_ids_to_check)]
#results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin(subject_ids_to_check)]

In [0]:
# Pull the 10 shortest unique notes from notes_enrolled4, restricted to subjects that also appear in results_GPT_full_df

valid_subjects = results_GPT_full_df['Subject ID'].unique()

filtered_notes = notes_enrolled4[
    notes_enrolled4['Subject ID'].isin(valid_subjects)
].dropna(subset=['NOTE_TEXT']).drop_duplicates(subset=['NOTE_TEXT']).copy()

filtered_notes['char_count'] = filtered_notes['NOTE_TEXT'].str.len()

smallest_10 = filtered_notes.nsmallest(10, 'char_count')[['Subject ID', 'NOTE_TEXT', 'char_count']]
display(smallest_10)

In [0]:
manual_score = manual_score[~manual_score['Subject ID'].isin([5 WES IDs])]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin([5 WES IDs])]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin([5 WES IDs])]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin([5 WES IDs])]

In [0]:
# Drop rows from manual_score that don't have a matching Subject ID in results_GPT_full_df

gpt_ids = set(results_GPT_full_df['Subject ID'].unique())

before_count = len(manual_score)
manual_score = manual_score[manual_score['Subject ID'].isin(gpt_ids)].reset_index(drop=True)
after_count = len(manual_score)

print(f"Rows before: {before_count}")
print(f"Rows after: {after_count}")
print(f"Rows dropped: {before_count - after_count}")

In [0]:
# Find all notes (in notes_enrolled4) belonging to subjects present in any of the
# three model result sets, where the note text contains three consecutive asterisks ("***")

all_model_subject_ids = (
    set(results_GPT_full_df['Subject ID'])
    | set(results_mini_full_df['Subject ID'])
    | set(results_claude_full_df['Subject ID'])
)

notes_subset = notes_enrolled4[
    notes_enrolled4['Subject ID'].isin(all_model_subject_ids)
].dropna(subset=['NOTE_TEXT']).copy()

triple_asterisk_notes = notes_subset[
    notes_subset['NOTE_TEXT'].str.contains(r'\*\*\*', regex=True, na=False)
][['Subject ID', 'NOTE_TEXT']].reset_index(drop=True)

print(f"Notes containing '***': {len(triple_asterisk_notes)}")

if triple_asterisk_notes.empty:
    print("No matching notes found.")
else:
    display(triple_asterisk_notes)

In [0]:
# Viewing score totals across entire dataframes (no subsetting)

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    if agg == 'count':
        score_totals[name] = df[score_col].count()
    else:
        score_totals[name] = df[score_col].sum()

print("Total phenotypes scored positive, per model, across all subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
print(f"notes_enrolled4 subjects: {notes_enrolled4['Subject ID'].nunique()}")
print(f"results_GPT_full_df subjects: {results_GPT_full_df['Subject ID'].nunique()}")
print(f"results_mini_full_df subjects: {results_mini_full_df['Subject ID'].nunique()}")
print(f"results_claude_full_df subjects: {results_claude_full_df['Subject ID'].nunique()}")
print(f"manual_score subjects: {manual_score['Subject ID'].nunique()}")

In [0]:
# Create sets of (Subject ID, Phenotype) tuples for each model
gpt_pairs = set(zip(results_GPT_full_df['Subject ID'], results_GPT_full_df['Phenotype']))
mini_pairs = set(zip(results_mini_full_df['Subject ID'], results_mini_full_df['Phenotype']))
claude_pairs = set(zip(results_claude_full_df['Subject ID'], results_claude_full_df['Phenotype']))

# Check if all three are identical
print(f"All three models have identical Subject ID/Phenotype pairs: {gpt_pairs == mini_pairs == claude_pairs}")

# If not, see what's different
print("In GPT but not Mini:", gpt_pairs - mini_pairs)
print("In Mini but not GPT:", mini_pairs - gpt_pairs)
print("In GPT but not Claude:", gpt_pairs - claude_pairs)
print("In Claude but not GPT:", claude_pairs - gpt_pairs)

In [0]:
# Finding parse error details
parse_error_ids = results_claude_full_df[results_claude_full_df['Phenotype'] == 'PARSE ERROR']['Subject ID']
print(len(parse_error_ids))
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(parse_error_ids)]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin(parse_error_ids)]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin(parse_error_ids)]
manual_score = manual_score[~manual_score['Subject ID'].isin(parse_error_ids)]

In [0]:
# Identify Subject IDs with Enrollment Status == 'Cancelled WES Testing' in notes_enrolled4,
# then drop those Subject IDs from notes_enrolled4, manual_score, and the three model result dataframes

cancelled_ids = set(
    notes_enrolled4.loc[notes_enrolled4['Enrollment Status'] == 'Cancelled WES Testing', 'Subject ID']
)

print(f"Subject IDs with Enrollment Status = 'Cancelled WES Testing': {len(cancelled_ids)}")
print(cancelled_ids)

before_counts = {
    'notes_enrolled4': notes_enrolled4['Subject ID'].nunique(),
    'manual_score': manual_score['Subject ID'].nunique(),
    'results_GPT_full_df': results_GPT_full_df['Subject ID'].nunique(),
    'results_mini_full_df': results_mini_full_df['Subject ID'].nunique(),
    'results_claude_full_df': results_claude_full_df['Subject ID'].nunique(),
}

notes_enrolled4 = notes_enrolled4[~notes_enrolled4['Subject ID'].isin(cancelled_ids)]
manual_score = manual_score[~manual_score['Subject ID'].isin(cancelled_ids)]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(cancelled_ids)]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin(cancelled_ids)]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin(cancelled_ids)]

after_counts = {
    'notes_enrolled4': notes_enrolled4['Subject ID'].nunique(),
    'manual_score': manual_score['Subject ID'].nunique(),
    'results_GPT_full_df': results_GPT_full_df['Subject ID'].nunique(),
    'results_mini_full_df': results_mini_full_df['Subject ID'].nunique(),
    'results_claude_full_df': results_claude_full_df['Subject ID'].nunique(),
}

print("\nUnique Subject IDs before -> after:")
for k in before_counts:
    print(f"  {k}: {before_counts[k]} -> {after_counts[k]} (dropped {before_counts[k] - after_counts[k]})")

In [0]:
print(f"results_GPT_full_df subjects: {results_GPT_full_df['Subject ID'].nunique()}")
print(f"results_mini_full_df subjects: {results_mini_full_df['Subject ID'].nunique()}")
print(f"results_claude_full_df subjects: {results_claude_full_df['Subject ID'].nunique()}")
print(f"manual_score subjects: {manual_score['Subject ID'].nunique()}")

In [0]:
# Phenotypes in manual_score but not GPT results
manual_score_not_in_GPT = manual_score[~manual_score['Phenotype'].isin(results_GPT_full_df['Phenotype'])]
print(manual_score_not_in_GPT)

In [0]:
# Removing phenotypes in manual_score that aren't in the GPT results. 
manual_score = manual_score[manual_score['Phenotype'].isin(results_GPT_full_df['Phenotype'])]

In [0]:
# Seeing which Subject IDs are in each AI but not the manual score
print("In GPT but not manual_score:", set(results_GPT_full_df['Subject ID']) - set(manual_score['Subject ID']))
print("In Mini but not manual_score:", set(results_mini_full_df['Subject ID']) - set(manual_score['Subject ID']))
print("In Claude but not manual_score:", set(results_claude_full_df['Subject ID']) - set(manual_score['Subject ID']))

In [0]:
# Subject IDs in notes_enrolled4 but not in manuaL_score
in_notes_not_manual = set(notes_enrolled4['Subject ID']) - set(manual_score['Subject ID'])
print(f"Subject IDs in notes_enrolled4 but not manual_score: {len(in_notes_not_manual)}")
print(in_notes_not_manual)

In [0]:
# Subject IDs in results_GPT_full_df but not in manual_score
gpt_not_manual_ids = set(results_GPT_full_df['Subject ID']) - set(manual_score['Subject ID'])

subset = notes_enrolled4[notes_enrolled4['Subject ID'].isin(gpt_not_manual_ids)]

if subset.empty:
    print("No subjects found in results_GPT_full_df that are missing from manual_score.")
else:
    display(subset)

In [0]:
#Combining all AI scoring to one dataframe

combined_df = results_GPT_full_df.merge(
    results_mini_full_df[['Subject ID', 'Phenotype', 'GPT_Mini_Score']],
    on=['Subject ID', 'Phenotype']
).merge(
    results_claude_full_df[['Subject ID', 'Phenotype', 'Claude_Sonnet_Score']],
    on=['Subject ID', 'Phenotype']
)

display(combined_df)

In [0]:
# Normalizing manual_score to match format of combined_df/AI scoring

# Get all unique subject IDs and phenotypes
all_subject_ids = combined_df['Subject ID'].unique()
all_phenotypes = checklist_filtered['Term from checklist'].dropna().unique()

# Create a complete grid of every (Subject ID, Phenotype) combination
full_grid = pd.MultiIndex.from_product(
    [all_subject_ids, all_phenotypes], 
    names=['Subject ID', 'Phenotype']
).to_frame(index=False)

# Mark which combinations were manually scored as present
manual_score['Manual_Score'] = 1
full_grid = full_grid.merge(
    manual_score[['Subject ID', 'Phenotype', 'Manual_Score']], 
    on=['Subject ID', 'Phenotype'], 
    how='left'
)
full_grid['Manual_Score'] = full_grid['Manual_Score'].fillna(0).astype(int)

display(full_grid)
print(len(full_grid))

In [0]:
# Quality auditing the normalized manual score output

print("Unique Subject IDs in all_subject_ids:", len(all_subject_ids))
print("Unique Phenotypes in all_phenotypes:", len(all_phenotypes))
print(f"Expected grid size: {len(all_subject_ids)} x {len(all_phenotypes)} = {len(all_subject_ids) * len(all_phenotypes)}")

# Check for duplicates in manual_score after the merge
print("\nDuplicate (Subject ID, Phenotype) pairs in full_grid:")
print(full_grid.duplicated(subset=['Subject ID', 'Phenotype']).sum())

# Check if any phenotypes in manual_score aren't in checklist_filtered
manual_phens = set(manual_score['Phenotype'])
checklist_phens = set(checklist_filtered['Term from checklist'])
print("\nPhenotypes in manual_score but not in checklist_filtered:")
print(manual_phens - checklist_phens)

In [0]:
# Troubleshooting 126 phenotypes. Possibly null values?
print("Null phenotypes in manual_score:", manual_score['Phenotype'].isna().sum())
print("Null phenotypes in checklist_filtered:", checklist_filtered['Term from checklist'].isna().sum())

In [0]:
# Dropping null phenotype values from checklist_filtered
checklist_filtered = checklist_filtered.dropna(subset=['Term from checklist'])
print(f"Checklist phenotypes after dropping nulls: {len(checklist_filtered)}")

In [0]:
# Showing and dropping duplicate phenotypes
# Show the duplicate terms
print("Duplicate terms:")
print(checklist_filtered[checklist_filtered['Term from checklist'].duplicated(keep=False)]['Term from checklist'].unique())

# Deduplicate
checklist_filtered = checklist_filtered.drop_duplicates(subset='Term from checklist')
print(f"\nChecklist phenotypes after deduplication: {len(checklist_filtered)}")

In [0]:
# Find the duplicate phenotype in checklist_filtered
print("Duplicate phenotypes in checklist_filtered:")
print(checklist_filtered[checklist_filtered['Term from checklist'].duplicated(keep=False)]['Term from checklist'].unique())

# Find duplicate (Subject ID, Phenotype) pairs in manual_score
print("\nDuplicate (Subject ID, Phenotype) pairs in manual_score:")
print(manual_score[manual_score.duplicated(subset=['Subject ID', 'Phenotype'], keep=False)])

## Building combined_df

In [0]:
#Rebuilding manual_score with clean checklist_filtered

# Recreate records from REDCap_mapping
records = []
for i in range(0, len(REDCap_mapping)):
    phen = []
    for j in range(0, len(REDCap_mapping.columns)):
        if pd.isnull(REDCap_mapping.iloc[i][j]) == False:
            phen.append(REDCap_mapping.iloc[i][j])
    records.append(phen)

records_formatted = pd.DataFrame(records)

# Recreate manual_list from records_formatted
manual_list = []
for j in range(0, len(records_formatted)):
    for i in range(0, len(records_formatted.columns)):
        manual_list.append(records_formatted[i][j])

manual_list = [x for x in manual_list if x is not None]

# Rebuild manual_score with clean checklist_filtered
rows = []
current_id = None
for item in manual_list:
    if item.startswith('WES'):
        current_id = item
    else:
        rows.append((current_id, item))

manual_df = pd.DataFrame(rows, columns=['Subject ID', 'Phenotype'])

manual_score = manual_df.merge(
    checklist_filtered[['Term from checklist', 'HPO term (ID)']],
    left_on='Phenotype',
    right_on='Term from checklist',
    how='inner'
).drop(columns=['Term from checklist'])

manual_score['Phenotype'] = manual_score['Phenotype'].str.strip()
print(f"manual_score rows: {len(manual_score)}")
print(f"Duplicate (Subject ID, Phenotype) pairs: {manual_score.duplicated(subset=['Subject ID', 'Phenotype']).sum()}")
display(manual_score)

In [0]:
# Normalizing the manual_score so it can be joined onto the combined dataframe 
# Get all unique subject IDs and phenotypes
all_subject_ids = combined_df['Subject ID'].unique()
all_phenotypes = checklist_filtered['Term from checklist'].unique()

# Create a complete grid of every (Subject ID, Phenotype) combination
full_grid = pd.MultiIndex.from_product(
    [all_subject_ids, all_phenotypes], 
    names=['Subject ID', 'Phenotype']
).to_frame(index=False)

# Mark which combinations were manually scored as present
manual_score['Manual_Score'] = 1
full_grid = full_grid.merge(
    manual_score[['Subject ID', 'Phenotype', 'Manual_Score']], 
    on=['Subject ID', 'Phenotype'], 
    how='left'
)
full_grid['Manual_Score'] = full_grid['Manual_Score'].fillna(0).astype(int)

print(len(full_grid))
print("Duplicate (Subject ID, Phenotype) pairs:", full_grid.duplicated(subset=['Subject ID', 'Phenotype']).sum())
display(full_grid)

In [0]:
# Adding the manual score values to the AI combined dataframe from earlier
combined_df = combined_df.merge(full_grid, on=['Subject ID', 'Phenotype'], how='left')
print(f"combined_df rows: {len(combined_df)}")
print(f"Null Manual_Score values: {combined_df['Manual_Score'].isna().sum()}")
display(combined_df)

## Disagreements With Manual

In [0]:
# Flag disagreements for each model vs manual
combined_df['GPT_disagree'] = (combined_df['GPT_Score'] != combined_df['Manual_Score']).astype(int)
combined_df['GPT_Mini_disagree'] = (combined_df['GPT_Mini_Score'] != combined_df['Manual_Score']).astype(int)
combined_df['Claude_Sonnet_disagree'] = (combined_df['Claude_Sonnet_Score'] != combined_df['Manual_Score']).astype(int)

# Total disagreements — any model disagrees with manual
combined_df['any_disagree'] = (
    (combined_df['GPT_disagree'] == 1) | 
    (combined_df['GPT_Mini_disagree'] == 1) | 
    (combined_df['Claude_Sonnet_disagree'] == 1)
).astype(int)

print(f"Total rows where at least one model disagrees with manual: {combined_df['any_disagree'].sum()}")

# Unique disagreements — only this model disagrees with manual, other two agree
combined_df['GPT_unique_disagree'] = (
    (combined_df['GPT_disagree'] == 1) & 
    (combined_df['GPT_Mini_disagree'] == 0) & 
    (combined_df['Claude_Sonnet_disagree'] == 0)
).astype(int)

combined_df['GPT_Mini_unique_disagree'] = (
    (combined_df['GPT_Mini_disagree'] == 1) & 
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['Claude_Sonnet_disagree'] == 0)
).astype(int)

combined_df['Claude_Sonnet_unique_disagree'] = (
    (combined_df['Claude_Sonnet_disagree'] == 1) & 
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['GPT_Mini_disagree'] == 0)
).astype(int)

print("\nTotal disagreements vs manual per model:")
print(f"  GPT: {combined_df['GPT_disagree'].sum()}")
print(f"  GPT Mini: {combined_df['GPT_Mini_disagree'].sum()}")
print(f"  Claude Sonnet: {combined_df['Claude_Sonnet_disagree'].sum()}")

print("\nUnique disagreements vs manual (only this model disagrees):")
print(f"  GPT: {combined_df['GPT_unique_disagree'].sum()}")
print(f"  GPT Mini: {combined_df['GPT_Mini_unique_disagree'].sum()}")
print(f"  Claude Sonnet: {combined_df['Claude_Sonnet_unique_disagree'].sum()}")

In [0]:
print("Unique values in Claude_Sonnet_Score:", combined_df['Claude_Sonnet_Score'].unique())
print("Unique values in GPT_Score:", combined_df['GPT_Score'].unique())
print("Unique values in GPT_Mini_Score:", combined_df['GPT_Mini_Score'].unique())
print("Unique values in Manual_Score:", combined_df['Manual_Score'].unique())

In [0]:
# GPT agrees but Mini disagrees
gpt_agree_mini_disagree = (
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['GPT_Mini_disagree'] == 1)
).sum()

# GPT agrees but Claude disagrees
gpt_agree_claude_disagree = (
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['Claude_Sonnet_disagree'] == 1)
).sum()

print(f"GPT agrees, Mini disagrees: {gpt_agree_mini_disagree}")
print(f"GPT agrees, Claude disagrees: {gpt_agree_claude_disagree}")

In [0]:
import matplotlib
print(matplotlib.__version__)
print(matplotlib.__file__)

In [0]:
# Scoring Venn Diagram
import matplotlib
print(matplotlib.__version__)  # should show 3.8.4

import matplotlib.pyplot as plt
from matplotlib_venn import venn3

gpt = set(combined_df[combined_df['GPT_disagree'] == 1].index)
mini = set(combined_df[combined_df['GPT_Mini_disagree'] == 1].index)
claude = set(combined_df[combined_df['Claude_Sonnet_disagree'] == 1].index)

# Compute actual subset sizes but normalize to equal circle sizes
subsets = (
    len(gpt - mini - claude),
    len(mini - gpt - claude),
    len(gpt & mini - claude),
    len(claude - gpt - mini),
    len(gpt & claude - mini),
    len(mini & claude - gpt),
    len(gpt & mini & claude),
)

# Use equal values to force equal circle sizes, but label with real counts
equal_subsets = (1, 1, 1, 1, 1, 1, 1)

fig, ax = plt.subplots(figsize=(8, 8))

v = venn3(
    subsets=equal_subsets,
    set_labels=('GPT', 'GPT Mini', 'Claude Sonnet'),
    set_colors=('#4C9BE8', '#57C4A7', '#E87B4C'),
    alpha=0.5,
    ax=ax
)

# Overwrite labels with real counts
labels = list(subsets)
for i, label in enumerate(v.subset_labels):
    if label:
        label.set_text(str(labels[i]))
        label.set_fontsize(11)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(13)
        label.set_fontweight('bold')

ax.set_title('Disagreements with manual score', fontsize=15, pad=20)
plt.tight_layout()
plt.show()

In [0]:
# Precision and recall calculations
from sklearn.metrics import precision_score, recall_score

total_rows = len(combined_df)
total_disagree = (combined_df['GPT_disagree'] | combined_df['GPT_Mini_disagree'] | combined_df['Claude_Sonnet_disagree']).sum()
pct_disagree = total_disagree / total_rows * 100

print(f"Total rows: {total_rows}")
print(f"Total disagreements (any model): {total_disagree}")
print(f"Percent disagreements: {pct_disagree:.1f}%")

print("\n--- Using manual score as ground truth ---\n")

for model, col in [('GPT', 'GPT_Score'), ('GPT Mini', 'GPT_Mini_Score'), ('Claude Sonnet', 'Claude_Sonnet_Score')]:
    y_true = combined_df['Manual_Score']
    y_pred = combined_df[col].astype(int)

    TP = ((y_pred == 1) & (y_true == 1)).sum()
    TN = ((y_pred == 0) & (y_true == 0)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    FN = ((y_pred == 0) & (y_true == 1)).sum()

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0

    print(f"=== {model} ===")
    print(f"TP = {TP} (both manual and {model} scored a 1)")
    print(f"TN = {TN} (both manual and {model} scored a 0)")
    print(f"FP = {FP} ({model} scored a 1 but manual scored a 0)")
    print(f"FN = {FN} ({model} scored a 0 but manual scored a 1)")
    print(f"Precision = TP / (TP + FP) = {TP} / ({TP} + {FP}) = {precision:.3f}")
    print(f"Recall    = TP / (TP + FN) = {TP} / ({TP} + {FN}) = {recall:.3f}")
    print()

In [0]:
# Sensitivity, Specificity, and F1 calculations

from sklearn.metrics import f1_score

print("--- Using manual score as ground truth ---\n")

for model, col in [('GPT', 'GPT_Score'), ('GPT Mini', 'GPT_Mini_Score'), ('Claude Sonnet', 'Claude_Sonnet_Score')]:
    y_true = combined_df['Manual_Score']
    y_pred = combined_df[col].astype(int)

    TP = ((y_pred == 1) & (y_true == 1)).sum()
    TN = ((y_pred == 0) & (y_true == 0)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    FN = ((y_pred == 0) & (y_true == 1)).sum()

    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    f1 = f1_score(y_true, y_pred)

    print(f"=== {model} ===")
    print(f"TP = {TP} (both manual and {model} scored a 1)")
    print(f"TN = {TN} (both manual and {model} scored a 0)")
    print(f"FP = {FP} ({model} scored a 1 but manual scored a 0)")
    print(f"FN = {FN} ({model} scored a 0 but manual scored a 1)")
    print(f"Sensitivity = TP / (TP + FN) = {TP} / ({TP} + {FN}) = {sensitivity:.3f}")
    print(f"Specificity = TN / (TN + FP) = {TN} / ({TN} + {FP}) = {specificity:.3f}")
    print(f"F1          = 2*TP / (2*TP + FP + FN) = 2*{TP} / (2*{TP} + {FP} + {FN}) = {f1:.3f}")
    print()

In [0]:
# Number of subjects with a disagreement
disagree_by_subject = combined_df.groupby('Subject ID').agg(
    any_disagree=('any_disagree', 'sum')
).reset_index()

subjects_with_disagree = (disagree_by_subject['any_disagree'] > 0).sum()
subjects_no_disagree = (disagree_by_subject['any_disagree'] == 0).sum()

print(f"Total subjects: {len(disagree_by_subject)}")
print(f"Subjects with at least one disagreement: {subjects_with_disagree}")
print(f"Subjects with no disagreements: {subjects_no_disagree}")
print(f"Percent of subjects with at least one disagreement: {subjects_with_disagree / len(disagree_by_subject) * 100:.1f}%")

# Figures

## Venn Diagrams, Upset Plot

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "GPT 5.4 Mini"
score_col = "GPT_Mini_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)
model_only  = len(model_set - manual_set)
both        = len(manual_set & model_set)

# Use equal values to force equal circle sizes, label with real counts
subsets = (manual_only, model_only, both)
equal_subsets = (1, 1, 1)

fig, ax = plt.subplots(figsize=(7, 7))

v = venn2(
    subsets=equal_subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#57C4A7'),
    alpha=0.5,
    ax=ax
)

# Overwrite labels with real counts
labels = list(subsets)
for i, label in enumerate(v.subset_labels):
    if label:
        label.set_text(str(labels[i]))
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

# Precision / Recall text beneath the figure
precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.15, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "GPT 5.4"
score_col = "GPT_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)   # FN
model_only  = len(model_set - manual_set)   # FP
both        = len(manual_set & model_set)   # TP

precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

# Use equal values to force equal circle sizes, label with real counts
subsets = (manual_only, model_only, both)
equal_subsets = (1, 1, 1)

fig, ax = plt.subplots(figsize=(7, 8.5))

v = venn2(
    subsets=equal_subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#4C9BE8'),
    alpha=0.5,
    ax=ax
)

# Overwrite labels with real counts
labels = list(subsets)
for i, label in enumerate(v.subset_labels):
    if label:
        label.set_text(str(labels[i]))
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

# Precision / Recall text beneath the figure
precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.2, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")
print(f"score_col in use: {score_col}")
print(f"TP={both}, FP={model_only}, FN={manual_only}")

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "Claude Sonnet 4.6"
score_col = "Claude_Sonnet_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)
model_only  = len(model_set - manual_set)
both        = len(manual_set & model_set)

# Use equal values to force equal circle sizes, label with real counts
subsets = (manual_only, model_only, both)
equal_subsets = (1, 1, 1)

fig, ax = plt.subplots(figsize=(7, 7))

v = venn2(
    subsets=equal_subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#E87B4C'),
    alpha=0.5,
    ax=ax
)


# Overwrite labels with real counts
labels = list(subsets)
for i, label in enumerate(v.subset_labels):
    if label:
        label.set_text(str(labels[i]))
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

# Precision / Recall text beneath the figure
precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0
formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.15, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")
print(f"score_col in use: {score_col}")
print(f"TP={both}, FP={model_only}, FN={manual_only}")

In [0]:
# Scoring Venn Diagram

import matplotlib.pyplot as plt
from matplotlib_venn import venn3

gpt = set(combined_df[combined_df['GPT_disagree'] == 1].index)
mini = set(combined_df[combined_df['GPT_Mini_disagree'] == 1].index)
claude = set(combined_df[combined_df['Claude_Sonnet_disagree'] == 1].index)

# Real counts now drive circle size directly (no more equal_subsets)
subsets = (
    len(gpt - mini - claude),
    len(mini - gpt - claude),
    len(gpt & mini - claude),
    len(claude - gpt - mini),
    len(gpt & claude - mini),
    len(mini & claude - gpt),
    len(gpt & mini & claude),
)

fig, ax = plt.subplots(figsize=(8, 8))

v = venn3(
    subsets=subsets,
    set_labels=('GPT', 'GPT Mini', 'Claude Sonnet'),
    set_colors=('#4C9BE8', '#57C4A7', '#E87B4C'),
    alpha=0.5,
    ax=ax
)

for label in v.subset_labels:
    if label:
        label.set_fontsize(11)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(13)
        label.set_fontweight('bold')

ax.set_title('Disagreements with manual score', fontsize=15, pad=20)
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

gpt = set(combined_df[combined_df['GPT_disagree'] == 1].index)
mini = set(combined_df[combined_df['GPT_Mini_disagree'] == 1].index)
claude = set(combined_df[combined_df['Claude_Sonnet_disagree'] == 1].index)

# Define each intersection: (label tuple of which sets are TRUE, count)
intersections = [
    (('GPT',), len(gpt - mini - claude)),
    (('GPT Mini',), len(mini - gpt - claude)),
    (('Claude Sonnet',), len(claude - gpt - mini)),
    (('GPT', 'GPT Mini'), len(gpt & mini - claude)),
    (('GPT', 'Claude Sonnet'), len(gpt & claude - mini)),
    (('GPT Mini', 'Claude Sonnet'), len(mini & claude - gpt)),
    (('GPT', 'GPT Mini', 'Claude Sonnet'), len(gpt & mini & claude)),
]

# Sort largest to smallest (like sort_by='cardinality')
intersections.sort(key=lambda x: x[1], reverse=True)

sets = ['GPT', 'GPT Mini', 'Claude Sonnet']
counts = [c for _, c in intersections]
memberships = [members for members, _ in intersections]

fig = plt.figure(figsize=(10, 7))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1.2], hspace=0.05)

# Top: bar chart of intersection sizes
ax_bar = fig.add_subplot(gs[0])
bar_positions = range(len(counts))
ax_bar.bar(bar_positions, counts, color='#4C9BE8', width=0.6)

for i, c in enumerate(counts):
    ax_bar.text(i, c + max(counts) * 0.01, str(c), ha='center', fontsize=10)

ax_bar.set_xticks([])
ax_bar.set_ylabel('Disagreement count', fontsize=12)
ax_bar.set_title('Disagreements with manual score', fontsize=15, pad=15)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)
ax_bar.spines['bottom'].set_visible(False)

# Bottom: dot matrix showing set membership per bar
ax_matrix = fig.add_subplot(gs[1], sharex=ax_bar)

for i, members in enumerate(memberships):
    active_idx = [sets.index(s) for s in members]
    # faint dots for inactive sets
    for j in range(len(sets)):
        color = '#333333' if j in active_idx else '#dddddd'
        ax_matrix.plot(i, j, 'o', color=color, markersize=10)
    # connecting line between active dots
    if len(active_idx) > 1:
        ax_matrix.plot([i, i], [min(active_idx), max(active_idx)], color='#333333', linewidth=1.5, zorder=0)

ax_matrix.set_yticks(range(len(sets)))
ax_matrix.set_yticklabels(sets, fontsize=12)
ax_matrix.set_xticks([])
ax_matrix.set_ylim(-0.5, len(sets) - 0.5)
ax_matrix.invert_yaxis()
for spine in ax_matrix.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "GPT 5.4 Mini"
score_col = "GPT_Mini_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)
model_only  = len(model_set - manual_set)
both        = len(manual_set & model_set)

precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

# Real counts now drive circle size directly (no more equal_subsets)
subsets = (manual_only, model_only, both)

this_total = manual_only + model_only + both

totals_by_model = {
    'GPT_Score': len(manual_set - set(combined_df[combined_df['GPT_Score'] == 1].index)) +
                 len(set(combined_df[combined_df['GPT_Score'] == 1].index) - manual_set) +
                 len(manual_set & set(combined_df[combined_df['GPT_Score'] == 1].index)),
    'GPT_Mini_Score': len(manual_set - set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)) +
                       len(set(combined_df[combined_df['GPT_Mini_Score'] == 1].index) - manual_set) +
                       len(manual_set & set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)),
    'Claude_Sonnet_Score': len(manual_set - set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)) +
                             len(set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index) - manual_set) +
                             len(manual_set & set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)),
}
max_total = max(totals_by_model.values())
normalize_to = this_total / max_total

fig, ax = plt.subplots(figsize=(7, 7))

v = venn2(
    subsets=subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#57C4A7'),
    alpha=0.5,
    normalize_to=normalize_to,   # <-- add this line
    ax=ax
)

for label in v.subset_labels:
    if label:
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.1, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")

In [0]:
print(f"{model}: this_total = {this_total}, normalize_to = {normalize_to:.4f}")

In [0]:
print(f"Manual constant check: {len(manual_set)} / {max_total} = {len(manual_set)/max_total:.5f}")

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "GPT 5.4"
score_col = "GPT_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)
model_only  = len(model_set - manual_set)
both        = len(manual_set & model_set)

precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

# Real counts now drive circle size directly (no more equal_subsets)
subsets = (manual_only, model_only, both)

this_total = manual_only + model_only + both

totals_by_model = {
    'GPT_Score': len(manual_set - set(combined_df[combined_df['GPT_Score'] == 1].index)) +
                 len(set(combined_df[combined_df['GPT_Score'] == 1].index) - manual_set) +
                 len(manual_set & set(combined_df[combined_df['GPT_Score'] == 1].index)),
    'GPT_Mini_Score': len(manual_set - set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)) +
                       len(set(combined_df[combined_df['GPT_Mini_Score'] == 1].index) - manual_set) +
                       len(manual_set & set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)),
    'Claude_Sonnet_Score': len(manual_set - set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)) +
                             len(set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index) - manual_set) +
                             len(manual_set & set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)),
}
max_total = max(totals_by_model.values())
normalize_to = this_total / max_total

fig, ax = plt.subplots(figsize=(7, 7))

v = venn2(
    subsets=subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#4C9BE8'),
    alpha=0.5,
    normalize_to=normalize_to,   # <-- add this line
    ax=ax
)

for label in v.subset_labels:
    if label:
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.1, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")

In [0]:
print(f"{model}: this_total = {this_total}, normalize_to = {normalize_to:.4f}")

In [0]:
print(f"Manual constant check: {len(manual_set)} / {max_total} = {len(manual_set)/max_total:.5f}")

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "Claude Sonnet 4.6"
score_col = "Claude_Sonnet_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)
model_only  = len(model_set - manual_set)
both        = len(manual_set & model_set)

precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

# Real counts now drive circle size directly (no more equal_subsets)
subsets = (manual_only, model_only, both)

this_total = manual_only + model_only + both

totals_by_model = {
    'GPT_Score': len(manual_set - set(combined_df[combined_df['GPT_Score'] == 1].index)) +
                 len(set(combined_df[combined_df['GPT_Score'] == 1].index) - manual_set) +
                 len(manual_set & set(combined_df[combined_df['GPT_Score'] == 1].index)),
    'GPT_Mini_Score': len(manual_set - set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)) +
                       len(set(combined_df[combined_df['GPT_Mini_Score'] == 1].index) - manual_set) +
                       len(manual_set & set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)),
    'Claude_Sonnet_Score': len(manual_set - set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)) +
                             len(set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index) - manual_set) +
                             len(manual_set & set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)),
}
max_total = max(totals_by_model.values())
normalize_to = this_total / max_total

fig, ax = plt.subplots(figsize=(7, 7))

v = venn2(
    subsets=subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#E87B4C'),
    alpha=0.5,
    normalize_to=normalize_to,   # <-- add this line
    ax=ax
)

for label in v.subset_labels:
    if label:
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.1, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")

In [0]:
print(f"{model}: this_total = {this_total}, normalize_to = {normalize_to:.4f}")

In [0]:
print(f"Manual constant check: {len(manual_set)} / {max_total} = {len(manual_set)/max_total:.5f}")

In [0]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

score_cols_map = {
    'GPT 5.4': 'GPT_Score',
    'GPT 5.4 Mini': 'GPT_Mini_Score',
    'Claude Sonnet 4.6': 'Claude_Sonnet_Score',
}
color_map = {
    'GPT 5.4': '#4C9BE8',
    'GPT 5.4 Mini': '#57C4A7',
    'Claude Sonnet 4.6': '#E87B4C',
}

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)

# --- Compute all three models' data first, so we know shared totals/limits up front ---
results = {}
for model, score_col in score_cols_map.items():
    model_set = set(combined_df[combined_df[score_col] == 1].index)
    manual_only = len(manual_set - model_set)
    model_only  = len(model_set - manual_set)
    both        = len(manual_set & model_set)
    this_total  = manual_only + model_only + both
    results[model] = {
        'score_col': score_col, 'manual_only': manual_only, 'model_only': model_only,
        'both': both, 'this_total': this_total,
        'precision': both / (both + model_only) if (both + model_only) > 0 else 0,
        'recall': both / (both + manual_only) if (both + manual_only) > 0 else 0,
    }

max_total = max(r['this_total'] for r in results.values())

# --- First pass: draw the largest-total model to capture reference axis limits ---
reference_model = max(results, key=lambda m: results[m]['this_total'])
r = results[reference_model]
normalize_to = r['this_total'] / max_total

fig_ref, ax_ref = plt.subplots(figsize=(7, 7))
venn2(
    subsets=(r['manual_only'], r['model_only'], r['both']),
    set_labels=('Manual', reference_model),
    set_colors=('#D4B896', color_map[reference_model]),
    alpha=0.5, normalize_to=normalize_to, ax=ax_ref
)
REFERENCE_XLIM = ax_ref.get_xlim()
REFERENCE_YLIM = ax_ref.get_ylim()
plt.close(fig_ref)  # discard this draft, we'll redraw all three properly below

# --- Second pass: draw all three with shared, fixed axis limits ---
for model, score_col in score_cols_map.items():
    r = results[model]
    normalize_to = r['this_total'] / max_total

    fig, ax = plt.subplots(figsize=(7, 7))

    v = venn2(
        subsets=(r['manual_only'], r['model_only'], r['both']),
        set_labels=('Manual', model),
        set_colors=('#D4B896', color_map[model]),
        alpha=0.5,
        normalize_to=normalize_to,
        ax=ax
    )

    ax.set_xlim(REFERENCE_XLIM)
    ax.set_ylim(REFERENCE_YLIM)
    ax.set_aspect('equal')

    for label in v.subset_labels:
        if label:
            label.set_fontsize(12)

    for patch in v.patches:
        if patch:
            patch.set_edgecolor('white')
            patch.set_linewidth(2)

    for label in v.set_labels:
        if label:
            label.set_fontsize(14)
            label.set_fontweight('bold')

    ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

    formula_text = (
        f"Precision = TP / (TP + FP) = {r['both']} / ({r['both']} + {r['model_only']}) = {r['precision']:.3f}\n"
        f"Recall    = TP / (TP + FN) = {r['both']} / ({r['both']} + {r['manual_only']}) = {r['recall']:.3f}"
    )
    fig.text(0.5, 0.1, formula_text, ha='center', va='bottom',
              fontsize=15, fontfamily='monospace', linespacing=1.8)

    plt.subplots_adjust(bottom=0.18)
    plt.show()

    print(f"Manual total positives: {len(manual_set)}")
    print(f"{model} total positives: {r['manual_only'] + r['both'] + (r['model_only'])}")  # note: model total = model_only + both
    print(f"Manual only (FN): {r['manual_only']}")
    print(f"{model} only (FP): {r['model_only']}")
    print(f"Both (TP): {r['both']}\n")

## Graphs

In [0]:
notes_enrolled4.columns

In [0]:
display(combined_df.head())

In [0]:
# Check for duplicate Subject IDs in notes_enrolled4
dup_check = notes_enrolled4['Subject ID'].duplicated().sum()
print(f"Total rows in notes_enrolled4: {len(notes_enrolled4)}")
print(f"Unique Subject IDs: {notes_enrolled4['Subject ID'].nunique()}")
print(f"Duplicate Subject ID rows: {dup_check}")

In [0]:
combined_df_full_info = combined_df.merge(
    notes_enrolled4,
    on='Subject ID',
    how='left'
)

print(f"combined_df rows: {len(combined_df)}")
print(f"combined_df_full_info rows: {len(combined_df_full_info)}")
print(f"Row count matches: {len(combined_df) == len(combined_df_full_info)}")
print(f"Null values introduced by merge (any column): {combined_df_full_info.isna().sum().sum() - combined_df.isna().sum().sum()}")

display(combined_df_full_info)

In [0]:
print(combined_df_full_info.shape)
print(combined_df_full_info['NOTE_TEXT'].duplicated().sum())
print(combined_df_full_info['NOTE_TEXT'].nunique())

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Deduplicate first, then compute character counts on unique notes only
unique_notes = combined_df_full_info['NOTE_TEXT'].dropna().drop_duplicates()
char_counts = unique_notes.str.len()

print(f"Total notes with text: {len(char_counts)}")
print(f"Min: {char_counts.min()}")
print(f"Max: {char_counts.max()}")
print(f"Mean: {char_counts.mean():.0f}")
print(f"Median: {char_counts.median():.0f}")
print(f"Std dev: {char_counts.std():.0f}")

# Fixed bin edges: equal-width bins from 0 to 25,000, plus one overflow bin
x_max = 25000
n_bins = 50
bin_edges = np.linspace(0, x_max, n_bins + 1)
bin_edges = np.append(bin_edges, char_counts.max() + 1)  # overflow bin

n_over = (char_counts > x_max).sum()
print(f"Notes above {x_max:,} chars (folded into overflow bin): {n_over}")

# Compute counts manually (not via ax.hist) so every bin -- including the
# wide overflow bin -- can be drawn at the same visual width
counts, edges = np.histogram(char_counts, bins=bin_edges)
bin_positions = np.arange(len(counts))

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(bin_positions, counts, width=1.0, color='#4C9BE8', edgecolor='white')

# Label only every 5th regular bin to avoid overcrowding, plus the overflow bin
tick_step = 5
tick_positions = list(range(0, n_bins, tick_step)) + [n_bins]  # last index = overflow bin
tick_labels = [f'{int(edges[i]):,}' if i < n_bins else f'>{x_max:,}' for i in tick_positions]

ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right')

ax.set_xlabel('Character count', fontsize=12)
ax.set_ylabel('Number of notes', fontsize=12)
ax.set_title('Distribution of NOTE_TEXT character counts', fontsize=14, pad=15)
ax.set_ylim(bottom=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = combined_df_full_info.dropna(subset=['NOTE_TEXT']).copy()
df['char_count'] = df['NOTE_TEXT'].str.len()

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {
    'Manual_Score': '#D4B896',
    'GPT_Score': '#4C9BE8',
    'GPT_Mini_Score': '#57C4A7',
    'Claude_Sonnet_Score': '#E87B4C',
}
labels = {
    'Manual_Score': 'Manual',
    'GPT_Score': 'GPT 5.4',
    'GPT_Mini_Score': 'GPT 5.4 Mini',
    'Claude_Sonnet_Score': 'Claude Sonnet 4.6',
}

subject_level = df.groupby('Subject ID').agg(
    char_count=('char_count', 'first'),
    Manual_Score=('Manual_Score', 'sum'),
    GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'),
    Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()

# Fixed bin edges: equal-width bins from actual min to 25,000, plus one overflow bin
x_min = subject_level['char_count'].min()
x_max = 25000
n_bins = 10
bin_edges = np.linspace(x_min, x_max, n_bins + 1)
bin_edges = np.append(bin_edges, subject_level['char_count'].max() + 1)  # overflow bin

subject_level['char_bin'] = pd.cut(subject_level['char_count'], bins=bin_edges, include_lowest=True)

binned = subject_level.groupby('char_bin', observed=True)[score_cols].mean()
bin_counts = subject_level.groupby('char_bin', observed=True).size()

# Evenly-spaced categorical positions (0, 1, 2, ...) instead of numeric midpoints —
# this makes it visually clear the x-axis is binned/discrete, not continuous
bin_positions = np.arange(len(binned))

# Label each bin with its actual range, e.g. "2,443–4,700"
tick_labels = []
for interval in binned.index:
    left = int(round(interval.left))
    if interval.right > x_max:
        tick_labels.append(f'{left:,}+')
    else:
        right = int(round(interval.right))
        tick_labels.append(f'{left:,}\u2013{right:,}')

print("Subjects per bin (check for sparse bins):")
print(bin_counts)

fig, ax = plt.subplots(figsize=(11, 6.5))

for col in score_cols:
    ax.plot(bin_positions, binned[col].values, marker='o', label=labels[col], color=colors[col], linewidth=2)

ax.set_xticks(bin_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right')

ax.set_xlabel('Note character count (binned)', fontsize=12)
ax.set_ylabel('Average phenotypes scored per subject', fontsize=12)
ax.set_title('Average phenotypes scored by note length', fontsize=15, pad=15)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.text(0.5, -0.05, f"X-axis shows {n_bins} equal-width character-count ranges plus an overflow bin for notes above {x_max:,} characters. "
                       f"Bins are evenly spaced categorically, not scaled to character count.",
          ha='center', va='top', fontsize=10, style='italic', color='#555555')

plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(binned)

In [0]:
# Pull the 10 unique notes with the smallest character counts, including Subject ID
unique_notes_df = combined_df_full_info.dropna(subset=['NOTE_TEXT']).drop_duplicates(subset=['NOTE_TEXT']).copy()
unique_notes_df['char_count'] = unique_notes_df['NOTE_TEXT'].str.len()

smallest_10 = unique_notes_df.nsmallest(10, 'char_count')[['Subject ID', 'NOTE_TEXT', 'char_count']]
display(smallest_10)

In [0]:
# How many rows per subject, and how many are exact duplicates?
rows_per_subject = combined_df_full_info.groupby('Subject ID').size()
print(rows_per_subject.describe())

# If you have a phenotype/category column, compare row count vs unique phenotype count per subject
if 'Phenotype' in combined_df_full_info.columns:
    check = combined_df_full_info.groupby('Subject ID').agg(
        total_rows=('Phenotype', 'size'),
        unique_phenotypes=('Phenotype', 'nunique')
    )
    print(check[check['total_rows'] != check['unique_phenotypes']])

In [0]:
# Confirm each subject has exactly one distinct NOTE_TEXT (and one char_count) across their 125 rows
notes_per_subject = combined_df_full_info.groupby('Subject ID')['NOTE_TEXT'].nunique()
print(notes_per_subject.value_counts())

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

df = combined_df_full_info.copy()
df['review_date'] = pd.to_datetime(df['Date of last chart review:'], errors='coerce')

n_unparsed = df['Date of last chart review:'].notna().sum() - df['review_date'].notna().sum()
print(f"Rows with a date that failed to parse: {n_unparsed}")

# Choose granularity: 'Q' for quarter, 'M' for month
period_freq = 'Q'   # change to 'M' for monthly
df['review_period'] = df['review_date'].dt.to_period(period_freq)

# ------------------------------------------------------------
# Figure 1: Distribution of notes by period
# ------------------------------------------------------------
subject_level = df.dropna(subset=['review_period']).drop_duplicates(subset='Subject ID')
notes_by_period = subject_level['review_period'].value_counts().sort_index()

fig1, ax1 = plt.subplots(figsize=(11, 5))
ax1.bar(notes_by_period.index.astype(str), notes_by_period.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel(f"Review date period ({'quarter' if period_freq=='Q' else 'month'})", fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by period of review date', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_period.values):
    ax1.text(i, v + notes_by_period.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Figure 2: Average phenotypes scored per note, by period, per model
# ------------------------------------------------------------
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']

subject_scores = df.dropna(subset=['review_period']).groupby('Subject ID').agg(
    review_period=('review_period', 'first'),
    Manual_Score=('Manual_Score', 'sum'),
    GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'),
    Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()

avg_by_period = subject_scores.groupby('review_period')[score_cols].mean()

colors = {
    'Manual_Score': '#D4B896',
    'GPT_Score': '#4C9BE8',
    'GPT_Mini_Score': '#57C4A7',
    'Claude_Sonnet_Score': '#E87B4C',
}
labels = {
    'Manual_Score': 'Manual',
    'GPT_Score': 'GPT 5.4',
    'GPT_Mini_Score': 'GPT 5.4 Mini',
    'Claude_Sonnet_Score': 'Claude Sonnet 4.6',
}

fig2, ax2 = plt.subplots(figsize=(11, 6))
x_labels = avg_by_period.index.astype(str)

for col in score_cols:
    ax2.plot(x_labels, avg_by_period[col], marker='o', label=labels[col], color=colors[col], linewidth=2)

ax2.set_xlabel(f"Review period ({'quarter' if period_freq=='Q' else 'month'})", fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by period of review date', fontsize=15, pad=15)
ax2.legend(frameon=False, fontsize=11)
ax2.set_ylim(bottom=0)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

display(notes_by_period)
display(avg_by_period)

In [0]:
notes_enrolled4.columns

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

df = combined_df_full_info.copy()
df['review_date'] = pd.to_datetime(df['NOTE_FILE_TIME_DTTM'], errors='coerce')

n_unparsed = df['NOTE_FILE_TIME_DTTM'].notna().sum() - df['review_date'].notna().sum()
print(f"Rows with a date that failed to parse: {n_unparsed}")

# Choose granularity: 'Q' for quarter, 'M' for month
period_freq = 'Q'   # change to 'M' for monthly
df['review_period'] = df['review_date'].dt.to_period(period_freq)

# ------------------------------------------------------------
# Figure 1: Distribution of notes by period
# ------------------------------------------------------------
subject_level = df.dropna(subset=['review_period']).drop_duplicates(subset='Subject ID')
notes_by_period = subject_level['review_period'].value_counts().sort_index()

fig1, ax1 = plt.subplots(figsize=(11, 5))
ax1.bar(notes_by_period.index.astype(str), notes_by_period.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel(f"Note file time period ({'quarter' if period_freq=='Q' else 'month'})", fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by period of note file time', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_period.values):
    ax1.text(i, v + notes_by_period.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Figure 2: Average phenotypes scored per note, by period, per model
# ------------------------------------------------------------
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']

subject_scores = df.dropna(subset=['review_period']).groupby('Subject ID').agg(
    review_period=('review_period', 'first'),
    Manual_Score=('Manual_Score', 'sum'),
    GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'),
    Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()

avg_by_period = subject_scores.groupby('review_period')[score_cols].mean()

colors = {
    'Manual_Score': '#D4B896',
    'GPT_Score': '#4C9BE8',
    'GPT_Mini_Score': '#57C4A7',
    'Claude_Sonnet_Score': '#E87B4C',
}
labels = {
    'Manual_Score': 'Manual',
    'GPT_Score': 'GPT 5.4',
    'GPT_Mini_Score': 'GPT 5.4 Mini',
    'Claude_Sonnet_Score': 'Claude Sonnet 4.6',
}

fig2, ax2 = plt.subplots(figsize=(11, 6))
x_labels = avg_by_period.index.astype(str)

for col in score_cols:
    ax2.plot(x_labels, avg_by_period[col], marker='o', label=labels[col], color=colors[col], linewidth=2)

ax2.set_xlabel(f"Note file time period ({'quarter' if period_freq=='Q' else 'month'})", fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by period of note file time', fontsize=15, pad=15)
ax2.legend(frameon=False, fontsize=11)
ax2.set_ylim(bottom=0)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

display(notes_by_period)
display(avg_by_period)

In [0]:
categorical_cols = ['CURRENT_AUTHOR', 'AUTHOR_ROLE', 'UPDATER', 'COSIGNER']

for col in categorical_cols:
    n_unique = combined_df_full_info[col].nunique(dropna=True)
    n_missing = combined_df_full_info[col].isna().sum()
    print(f"{col}: {n_unique} unique values, {n_missing} missing rows")
    print(combined_df_full_info[col].value_counts(dropna=True).head(15))
    print()

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['CURRENT_AUTHOR']).drop_duplicates(subset='Subject ID').copy()
top_authors = subject_level['CURRENT_AUTHOR'].value_counts().head(14).index.tolist()
subject_level['CURRENT_AUTHOR_grouped'] = subject_level['CURRENT_AUTHOR'].apply(lambda x: x if x in top_authors else 'Other')

order = top_authors + ['Other']
notes_by_cat = subject_level['CURRENT_AUTHOR_grouped'].value_counts().reindex(order).dropna()

fig1, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Current author (top 14 + other)', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by current author', fontsize=15, pad=15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['CURRENT_AUTHOR']).groupby('Subject ID').agg(
    CURRENT_AUTHOR=('CURRENT_AUTHOR', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
subject_scores['CURRENT_AUTHOR_grouped'] = subject_scores['CURRENT_AUTHOR'].apply(lambda x: x if x in top_authors else 'Other')
avg_by_cat = subject_scores.groupby('CURRENT_AUTHOR_grouped')[score_cols].mean().reindex(order).dropna()

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(13, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Current author (top 14 + other)', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by current author', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['AUTHOR_ROLE']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['AUTHOR_ROLE'].value_counts()

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Author role', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by author role', fontsize=15, pad=15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


subject_scores = df.dropna(subset=['AUTHOR_ROLE']).groupby('Subject ID').agg(
    AUTHOR_ROLE=('AUTHOR_ROLE', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('AUTHOR_ROLE')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(11, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Author role', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by author role', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['UPDATER']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['UPDATER'].value_counts()

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Updater', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by updater', fontsize=15, pad=15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['UPDATER']).groupby('Subject ID').agg(
    UPDATER=('UPDATER', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('UPDATER')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(11, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Updater', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by updater', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['COSIGNER']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['COSIGNER'].value_counts()

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Cosigner', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by cosigner', fontsize=15, pad=15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['COSIGNER']).groupby('Subject ID').agg(
    COSIGNER=('COSIGNER', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('COSIGNER')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(11, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Cosigner', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by cosigner', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# Collapse COSIGNER into a binary category: has a cosigner (non-null) vs. no cosigner (null)
df['COSIGNER_STATUS'] = np.where(df['COSIGNER'].notna(), 'Has Cosigner', 'No Cosigner')

subject_level = df.drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['COSIGNER_STATUS'].value_counts().reindex(['Has Cosigner', 'No Cosigner'])

fig1, ax1 = plt.subplots(figsize=(7, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Cosigner status', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by cosigner status', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

subject_scores = df.groupby('Subject ID').agg(
    COSIGNER_STATUS=('COSIGNER_STATUS', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('COSIGNER_STATUS')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(9, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Cosigner status', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by cosigner status', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=0)
ax2.set_ylim(bottom=0)
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
n_unique = notes_enrolled4['COSIGNER'].nunique(dropna=True)
n_missing = notes_enrolled4['COSIGNER'].isna().sum()

print(f"COSIGNER: {n_unique} unique values, {n_missing} missing rows")
print(notes_enrolled4['COSIGNER'].value_counts(dropna=True))

In [0]:
# ============================================================
# Variable-width age bins: 1-year bins up to 14, 2-year bins up to 20,
# 5-year bins up to 30, then 10-year bins beyond that
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

df['review_date'] = pd.to_datetime(df['Date of last chart review:'], errors='coerce')
df['dob'] = pd.to_datetime(df['D.O.B.'], errors='coerce')
df['age_at_note_years'] = (df['review_date'] - df['dob']).dt.days / 365.25

n_parsed = df[['review_date', 'dob']].notna().all(axis=1).sum()
print(f"Rows with both dates successfully parsed: {n_parsed} / {len(df)}")

# --- Build variable-width bin edges: 1yr up to 14, 2yr up to 20, 5yr up to 30, 10yr beyond ---
max_age = df['age_at_note_years'].max()

edges = list(range(0, 15, 1))                     # 0,1,2,...,14  -> 1-year bins
edges += list(range(16, 21, 2))                    # 16,18,20      -> 2-year bins (14-16,16-18,18-20)
edges += [25, 30]                                  # 25,30         -> 5-year bins (20-25,25-30)
next_edge = 40
while next_edge - 10 < max_age:
    edges.append(next_edge)                        # 40,50,60,...  -> 10-year bins
    next_edge += 10

edges = sorted(set(edges))
print(f"Bin edges: {edges}")

df['age_bin'] = pd.cut(df['age_at_note_years'], bins=edges, right=False, include_lowest=True)

subject_level = df.dropna(subset=['age_bin']).drop_duplicates(subset='Subject ID').copy()

# Figure 1: distribution of notes by age bin
notes_by_bin = subject_level['age_bin'].value_counts().sort_index()
notes_by_bin = notes_by_bin[notes_by_bin.index.isin(subject_level['age_bin'].cat.categories)]

# Evenly-spaced categorical positions; 1-year bins get a single-number label,
# wider bins keep a range label since two edges are needed to disambiguate them
bin_positions = np.arange(len(notes_by_bin))
tick_labels = []
for iv in notes_by_bin.index:
    width = iv.right - iv.left
    if width == 1:
        tick_labels.append(f'{int(iv.right)}')
    else:
        tick_labels.append(f'{int(iv.left)}\u2013{int(iv.right)}')

fig1, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(bin_positions, notes_by_bin.values, marker='o', color='#4C9BE8', linewidth=2)
ax1.set_xticks(bin_positions)
ax1.set_xticklabels(tick_labels, rotation=45, ha='right')
ax1.set_xlabel('Age at note (years, binned)', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by age at note', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
fig1.text(0.5, -0.05, "X-axis bins: 1-year intervals up to 14 (labeled by single age), 2-year intervals up to 20, "
                       "5-year intervals up to 30, then 10-year intervals (labeled by range). "
                       "Bins are evenly spaced categorically, not scaled to age.",
          ha='center', va='top', fontsize=10, style='italic', color='#555555')
plt.tight_layout()
plt.show()

# Figure 2: average phenotypes scored per note, by age bin, per model
subject_scores = df.dropna(subset=['age_bin']).groupby('Subject ID').agg(
    age_bin=('age_bin', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_bin = subject_scores.groupby('age_bin', observed=True)[score_cols].mean()
avg_by_bin = avg_by_bin.reindex(notes_by_bin.index)

x_vals = bin_positions

fig2, ax2 = plt.subplots(figsize=(13, 6))

# Extend the right margin so end-of-line labels have room to sit outside the plotted data
x_range = x_vals.max() - x_vals.min()
ax2.set_xlim(x_vals.min(), x_vals.max() + x_range * 0.18)

for col in score_cols:
    y_vals = avg_by_bin[col].values
    ax2.plot(x_vals, y_vals, marker='o', color=colors[col], linewidth=2)

    # Label directly at the last valid point of each line
    last_valid_idx = np.where(~np.isnan(y_vals))[0]
    if len(last_valid_idx) > 0:
        last_idx = last_valid_idx[-1]
        ax2.annotate(
            labels[col],
            xy=(x_vals[last_idx], y_vals[last_idx]),
            xytext=(8, 0), textcoords='offset points',
            va='center', fontsize=11, color=colors[col], fontweight='bold'
        )

ax2.set_xticks(bin_positions)
ax2.set_xticklabels(tick_labels, rotation=45, ha='right')
ax2.set_xlabel('Age at note (years, binned)', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by age at note', fontsize=15, pad=15)
ax2.set_ylim(bottom=0)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig2.text(0.5, -0.05, "X-axis bins: 1-year intervals up to 14 (labeled by single age), 2-year intervals up to 20, "
                       "5-year intervals up to 30, then 10-year intervals (labeled by range). "
                       "Bins are evenly spaced categorically, not scaled to age.",
          ha='center', va='top', fontsize=10, style='italic', color='#555555')

plt.tight_layout()
plt.show()

display(notes_by_bin)
display(avg_by_bin)

In [0]:
# ============================================================
# Variable-width age bins: 1-year bins up to 14, 2-year bins up to 20,
# 5-year bins up to 30, 10-year bins up to 70, then a single 70+ overflow bin
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

df['review_date'] = pd.to_datetime(df['Date of last chart review:'], errors='coerce')
df['dob'] = pd.to_datetime(df['D.O.B.'], errors='coerce')
df['age_at_note_years'] = (df['review_date'] - df['dob']).dt.days / 365.25

n_parsed = df[['review_date', 'dob']].notna().all(axis=1).sum()
print(f"Rows with both dates successfully parsed: {n_parsed} / {len(df)}")

# --- Build variable-width bin edges: 1yr up to 14, 2yr up to 20, 5yr up to 30,
#     10yr up to 70, then a single overflow bin for everything 70+ ---
max_age = df['age_at_note_years'].max()

edges = list(range(0, 15, 1))                     # 0,1,2,...,14  -> 1-year bins
edges += list(range(16, 21, 2))                    # 16,18,20      -> 2-year bins (14-16,16-18,18-20)
edges += [25, 30]                                  # 25,30         -> 5-year bins (20-25,25-30)
edges += list(range(40, 71, 10))                   # 40,50,60,70   -> 10-year bins (30-40,...,60-70)
edges.append(max(max_age + 1, 71))                 # final edge -> overflow bin covering 70+

edges = sorted(set(edges))
print(f"Bin edges: {edges}")

df['age_bin'] = pd.cut(df['age_at_note_years'], bins=edges, right=False, include_lowest=True)

subject_level = df.dropna(subset=['age_bin']).drop_duplicates(subset='Subject ID').copy()

# Figure 1: distribution of notes by age bin
notes_by_bin = subject_level['age_bin'].value_counts().sort_index()
notes_by_bin = notes_by_bin[notes_by_bin.index.isin(subject_level['age_bin'].cat.categories)]

# Evenly-spaced categorical positions; 1-year bins get a single-number label,
# wider bins get a range label, and the final bin is labeled "70+"
bin_positions = np.arange(len(notes_by_bin))
tick_labels = []
for iv in notes_by_bin.index:
    width = iv.right - iv.left
    if iv.left >= 70:
        tick_labels.append('70+')
    elif width == 1:
        tick_labels.append(f'{int(iv.right)}')
    else:
        tick_labels.append(f'{int(iv.left)}\u2013{int(iv.right)}')

fig1, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(bin_positions, notes_by_bin.values, marker='o', color='#4C9BE8', linewidth=2)
ax1.set_xticks(bin_positions)
ax1.set_xticklabels(tick_labels, rotation=45, ha='right')
ax1.set_xlabel('Age at note (years, binned)', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by age at note', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
fig1.text(0.5, -0.05, "X-axis bins: 1-year intervals up to 14, 2-year intervals up to 20, 5-year intervals up to 30, "
                       "10-year intervals up to 70, then a single 70+ bin. "
                       "Bins are evenly spaced categorically, not scaled to age.",
          ha='center', va='top', fontsize=10, style='italic', color='#555555')
plt.tight_layout()
plt.show()

# Figure 2: average phenotypes scored per note, by age bin, per model
subject_scores = df.dropna(subset=['age_bin']).groupby('Subject ID').agg(
    age_bin=('age_bin', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_bin = subject_scores.groupby('age_bin', observed=True)[score_cols].mean()
avg_by_bin = avg_by_bin.reindex(notes_by_bin.index)

x_vals = bin_positions

fig2, ax2 = plt.subplots(figsize=(12, 6))

for col in score_cols:
    y_vals = avg_by_bin[col].values
    ax2.plot(x_vals, y_vals, marker='o', label=labels[col], color=colors[col], linewidth=2)

ax2.set_xticks(bin_positions)
ax2.set_xticklabels(tick_labels, rotation=45, ha='right')
ax2.set_xlabel('Age at note (years, binned)', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by age at note', fontsize=15, pad=15)
ax2.set_ylim(bottom=0)
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig2.text(0.5, -0.05, "X-axis bins: 1-year intervals up to 14, 2-year intervals up to 20, 5-year intervals up to 30, "
                       "10-year intervals up to 70, then a single 70+ bin. "
                       "Bins are evenly spaced categorically, not scaled to age.",
          ha='center', va='top', fontsize=10, style='italic', color='#555555')

plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_bin)
display(avg_by_bin)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['Enrollment Status']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['Enrollment Status'].value_counts()

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Enrollment status', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by enrollment status', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['Enrollment Status']).groupby('Subject ID').agg(
    **{'Enrollment Status': ('Enrollment Status', 'first')},
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('Enrollment Status')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(11, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Enrollment status', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by enrollment status', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.set_ylim(bottom=0)
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['NOTE_TYPE']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['NOTE_TYPE'].value_counts()

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Note type', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by note type', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['NOTE_TYPE']).groupby('Subject ID').agg(
    **{'NOTE_TYPE': ('NOTE_TYPE', 'first')},
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('NOTE_TYPE')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(11, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Note type', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by note type', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.set_ylim(bottom=0)
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = combined_df_full_info.copy()

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_scores = df.groupby('Subject ID').agg(
    Manual_Score=('Manual_Score', 'sum'),
    GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'),
    Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()

print(subject_scores[score_cols].describe())

# One bin per integer value, so bars align exactly with whole-number scores
all_vals = pd.concat([subject_scores[col].dropna() for col in score_cols])
bin_edges = np.arange(all_vals.min(), all_vals.max() + 2) - 0.5  # centers bars on integers

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
axes = axes.flatten()

for ax, col in zip(axes, score_cols):
    ax.hist(subject_scores[col].dropna(), bins=bin_edges, color=colors[col], edgecolor='white')
    ax.set_title(labels[col], fontsize=13)
    ax.set_ylim(bottom=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', labelbottom=True)  # force x tick labels on every subplot

fig.supxlabel('Phenotypes scored per subject', fontsize=12)
fig.supylabel('Number of subjects', fontsize=12)
fig.suptitle('Distribution of phenotypes scored, by model', fontsize=15)

plt.tight_layout()
plt.show()

In [0]:
!pip install openpyxl pandas



file_path = "/Workspace/Users/user/user/UMD Enrollment Log_11.7.2025_for_Gabriel 2.xlsx"

umd_enrollment_log = pd.read_excel(file_path)
print(umd_enrollment_log.columns)

In [0]:
# Quality check: confirm each Subject ID maps to exactly one unique
# 'Consented By:' value after the merge (i.e. no subject has conflicting values)

consented_per_subject = umd_enrollment_log.dropna(subset=['Consented By:']).groupby('Subject ID')['Consented By:'].nunique()

n_multi = (consented_per_subject > 1).sum()
print(f"Subjects with more than one distinct 'Consented By:' value: {n_multi}")

if n_multi > 0:
    problem_subjects = consented_per_subject[consented_per_subject > 1].index
    print("Subjects with conflicting values:")
    display(
        umd_enrollment_log[umd_enrollment_log['Subject ID'].isin(problem_subjects)][['Subject ID', 'Consented By:']]
        .drop_duplicates()
        .sort_values('Subject ID')
    )
else:
    print("Every subject has exactly one unique 'Consented By:' value. Merge looks clean.")

# Subjects with no matching 'Consented By:' value after the merge (i.e. didn't match
# to anything in umd_enrollment_log, or matched but the field itself was blank)

subjects_no_consent = umd_enrollment_log[umd_enrollment_log['Consented By:'].isna()][['Subject ID']].drop_duplicates().sort_values('Subject ID')

print(f"Subjects with no 'Consented By:' value: {len(subjects_no_consent)}")
display(subjects_no_consent)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Join 'Consented By:' from the enrollment log onto combined_df_full_info, keyed on Subject ID
df = combined_df_full_info.merge(
    umd_enrollment_log[['Subject ID', 'Consented By:']],
    on='Subject ID',
    how='left'
)

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

print(f"Rows with no matching 'Consented By:' after join: {df['Consented By:'].isna().sum()} / {len(df)}")

subject_level = df.dropna(subset=['Consented By:']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['Consented By:'].value_counts().sort_values(ascending=True)  # ascending for barh (largest on top)

fig1, ax1 = plt.subplots(figsize=(9, max(4, 0.4 * len(notes_by_cat))))
ax1.barh(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_ylabel('Consented by', fontsize=12)
ax1.set_xlabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by consented by', fontsize=15, pad=15)
ax1.set_xlim(left=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(v + notes_by_cat.max() * 0.01, i, str(v), va='center', ha='left', fontsize=10)
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['Consented By:']).groupby('Subject ID').agg(
    **{'Consented By:': ('Consented By:', 'first')},
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('Consented By:')[score_cols].mean().reindex(notes_by_cat.index)  # same sorted order

y = np.arange(len(avg_by_cat))
height = 0.2
fig2, ax2 = plt.subplots(figsize=(11, max(5, 0.6 * len(avg_by_cat))))
for i, col in enumerate(score_cols):
    ax2.barh(y + (i - 1.5) * height, avg_by_cat[col], height, label=labels[col], color=colors[col])

ax2.set_ylabel('Consented by', fontsize=12)
ax2.set_xlabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by consented by', fontsize=15, pad=15)
ax2.set_yticks(y)
ax2.set_yticklabels(avg_by_cat.index.astype(str))
ax2.set_xlim(left=0)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=4, frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

display(notes_by_cat.sort_values(ascending=False))
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()

model_cols = ['GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# For each row, flag disagreement between each model and Manual
for col in model_cols:
    df[f'{col}_disagree'] = (df[col] != df['Manual_Score']).astype(int)

# Disagreement rate per phenotype, per model
disagreement_by_phenotype = df.groupby('Phenotype')[[f'{col}_disagree' for col in model_cols]].mean()
disagreement_by_phenotype.columns = model_cols  # rename back for convenience

# Sort by average disagreement across the three models, descending (most-disagreed at top)
disagreement_by_phenotype['avg_disagree'] = disagreement_by_phenotype[model_cols].mean(axis=1)
disagreement_by_phenotype = disagreement_by_phenotype.sort_values('avg_disagree', ascending=True)  # ascending for barh (highest on top)
disagreement_by_phenotype = disagreement_by_phenotype.drop(columns='avg_disagree')

y = np.arange(len(disagreement_by_phenotype))
height = 0.25

fig, ax = plt.subplots(figsize=(11, max(6, 0.35 * len(disagreement_by_phenotype))))
for i, col in enumerate(model_cols):
    ax.barh(y + (i - 1) * height, disagreement_by_phenotype[col], height,
            label=labels[col], color=colors[col])

ax.set_yticks(y)
ax.set_yticklabels(disagreement_by_phenotype.index.astype(str))
ax.set_xlabel('Disagreement rate with Manual score', fontsize=12)
ax.set_ylabel('Phenotype', fontsize=12)
ax.set_title('Disagreement with Manual scoring, by phenotype', fontsize=15, pad=15)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.06), ncol=3, frameon=False, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

display(disagreement_by_phenotype.sort_values(disagreement_by_phenotype.columns[0], ascending=False))

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
model_cols = ['GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# Row-level disagreement flags: each model vs Manual
for col in model_cols:
    df[f'{col}_disagree'] = (df[col] != df['Manual_Score']).astype(int)

# Average score (rate scored positive) per phenotype, per model -- for the bars
score_by_phenotype = df.groupby('Phenotype')[score_cols].mean()

# Disagreement rate per phenotype, per model -- used only for sorting
disagreement_by_phenotype = df.groupby('Phenotype')[[f'{col}_disagree' for col in model_cols]].mean()

# Sort by the single highest-disagreeing model per phenotype (max across the three models)
sort_key = disagreement_by_phenotype.max(axis=1)
score_by_phenotype = score_by_phenotype.reindex(sort_key.sort_values(ascending=True).index)  # ascending -> highest ends up at top of barh

y = np.arange(len(score_by_phenotype))
height = 0.2

fig, ax = plt.subplots(figsize=(11, max(6, 0.4 * len(score_by_phenotype))))
for i, col in enumerate(score_cols):
    ax.barh(y + (i - 1.5) * height, score_by_phenotype[col], height,
            label=labels[col], color=colors[col])

ax.set_yticks(y)
ax.set_yticklabels(score_by_phenotype.index.astype(str))
ax.set_xlabel('Rate scored positive', fontsize=12)
ax.set_ylabel('Phenotype', fontsize=12)
ax.set_title('Phenotype scoring rate, by model\n(sorted by highest-disagreeing model vs Manual)', fontsize=15, pad=15)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.06), ncol=4, frameon=False, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# Show scores alongside each model's disagreement rate, for reference
result = score_by_phenotype.copy()
for col in model_cols:
    result[f'{col}_disagreement'] = disagreement_by_phenotype.loc[result.index, f'{col}_disagree']
display(result.iloc[::-1])  # highest-disagreement phenotype first

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
model_cols = ['GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# Row-level disagreement flags: each model vs Manual
for col in model_cols:
    df[f'{col}_disagree'] = (df[col] != df['Manual_Score']).astype(int)

# Total count scored positive per phenotype, per model -- for the bars
score_by_phenotype = df.groupby('Phenotype')[score_cols].sum()

# Total disagreement count per phenotype, per model -- used only for sorting
disagreement_by_phenotype = df.groupby('Phenotype')[[f'{col}_disagree' for col in model_cols]].sum()

# Sort by the single highest-disagreeing model per phenotype (max across the three models)
sort_key = disagreement_by_phenotype.max(axis=1)
score_by_phenotype = score_by_phenotype.reindex(sort_key.sort_values(ascending=True).index)  # ascending -> highest ends up at top of barh

y = np.arange(len(score_by_phenotype))
height = 0.2

fig, ax = plt.subplots(figsize=(11, max(6, 0.4 * len(score_by_phenotype))))
for i, col in enumerate(score_cols):
    ax.barh(y + (i - 1.5) * height, score_by_phenotype[col], height,
            label=labels[col], color=colors[col])

ax.set_yticks(y)
ax.set_yticklabels(score_by_phenotype.index.astype(str))
ax.set_xlabel('Number of subjects scored positive', fontsize=12)
ax.set_ylabel('Phenotype', fontsize=12)
ax.set_title('Phenotype scoring counts, by model\n(sorted by highest-disagreeing model vs Manual)', fontsize=15, pad=15)
ax.set_xlim(left=0)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.06), ncol=4, frameon=False, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# Show scores alongside each model's disagreement count, for reference
result = score_by_phenotype.copy()
for col in model_cols:
    result[f'{col}_disagreement'] = disagreement_by_phenotype.loc[result.index, f'{col}_disagree']
display(result.iloc[::-1])  # highest-disagreement phenotype first

In [0]:
# Count of subjects where Manual scored 'Autism Spectrum Disorder (ASD)' as positive

asd_manual_count = df[
    (df['Phenotype'] == 'Autism Spectrum Disorder (ASD)') &
    (df['Manual_Score'] == 1)
]['Subject ID'].nunique()

total_asd_rows = (df['Phenotype'] == 'Autism Spectrum Disorder (ASD)').sum()

print(f"Subjects with 'Autism Spectrum Disorder (ASD)' scored positive by Manual: {asd_manual_count}")
print(f"Total subjects with an ASD phenotype row: {total_asd_rows}")

In [0]:
import requests
import json
import os

output_folder = "/Workspace/Users/user/user/"
hp_json_path = output_folder + "hp.json"

# One-time download of the full HPO ontology (official OBO Foundry release).
# Only run this once -- after this, everything is done locally from the saved file.
if not os.path.exists(hp_json_path):
    url = "http://purl.obolibrary.org/obo/hp.json"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    with open(hp_json_path, 'wb') as f:
        f.write(resp.content)
    print(f"Downloaded HPO ontology to {hp_json_path} ({len(resp.content)/1e6:.1f} MB)")
else:
    print(f"HPO ontology already downloaded at {hp_json_path}, skipping.")

In [0]:
# ------------------------------------------------------------
# Parse the local hp.json file -- no network calls below this point
# ------------------------------------------------------------
with open(hp_json_path, 'r') as f:
    hpo_data = json.load(f)

graph = hpo_data['graphs'][0]

def iri_to_short(iri):
    # e.g. "http://purl.obolibrary.org/obo/HP_0001250" -> "HP:0001250"
    frag = iri.rsplit('/', 1)[-1]
    return frag.replace('_', ':', 1)

def short_to_iri(short_id):
    # e.g. "HP:0001250" -> "http://purl.obolibrary.org/obo/HP_0001250"
    return "http://purl.obolibrary.org/obo/" + short_id.replace(':', '_', 1)

# id (short form) -> label
id_to_label = {
    iri_to_short(n['id']): n.get('lbl', '')
    for n in graph['nodes'] if 'lbl' in n
}

# child (short form) -> set of parent ids (short form), from is_a edges only
parent_map = {}
for e in graph['edges']:
    if e.get('pred') in ('is_a', 'http://www.w3.org/2000/01/rdf-schema#subClassOf'):
        child = iri_to_short(e['sub'])
        parent = iri_to_short(e['obj'])
        parent_map.setdefault(child, set()).add(parent)

# Top-level categories = direct children of HP:0000118 (Phenotypic abnormality)
top_level_ids = {
    child for child, parents in parent_map.items() if 'HP:0000118' in parents
}
print(f"Top-level HPO categories found: {len(top_level_ids)}")
for tid in sorted(top_level_ids):
    print(f"  {tid}: {id_to_label.get(tid)}")

def find_top_level_category(hpo_id, parent_map, top_level_ids, id_to_label):
    """BFS up the parent hierarchy until a top-level category is hit."""
    if hpo_id in top_level_ids:
        return id_to_label.get(hpo_id, hpo_id)
    visited = set()
    frontier = [hpo_id]
    while frontier:
        next_frontier = []
        for node in frontier:
            if node in visited:
                continue
            visited.add(node)
            for parent in parent_map.get(node, []):
                if parent in top_level_ids:
                    return id_to_label.get(parent, parent)
                next_frontier.append(parent)
        frontier = next_frontier
    return 'Unmapped / Other'

# Build the HPO ID -> category lookup for just the phenotypes you actually use
phenotype_to_hpo = (
    checklist_filtered[['Term from checklist', 'HPO term (ID)']]
    .dropna()
    .drop_duplicates(subset='Term from checklist')
    .set_index('Term from checklist')['HPO term (ID)']
    .to_dict()
)

hpo_to_category = {
    hpo_id: find_top_level_category(hpo_id, parent_map, top_level_ids, id_to_label)
    for hpo_id in set(phenotype_to_hpo.values())
}

print(f"\nMapped {len(hpo_to_category)} unique HPO IDs to categories.")
display(pd.DataFrame(hpo_to_category.items(), columns=['HPO_ID', 'HPO_Category']))

In [0]:
# ============================================================
# Chart 1: Distribution of phenotypes by parent (HPO) ontology category
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

# Count distinct phenotypes per HPO category (using the mapping built earlier)
phenotype_category_df = pd.DataFrame([
    {'Phenotype': pheno, 'HPO_Category': hpo_to_category.get(hpo_id, 'Unmapped / Other')}
    for pheno, hpo_id in phenotype_to_hpo.items()
])
# Include any phenotypes that had no HPO ID match at all in checklist_filtered
unmapped_phenos = set(df['Phenotype'].unique()) - set(phenotype_to_hpo.keys())
if unmapped_phenos:
    phenotype_category_df = pd.concat([
        phenotype_category_df,
        pd.DataFrame({'Phenotype': list(unmapped_phenos), 'HPO_Category': 'Unmapped / Other'})
    ], ignore_index=True)

phenotypes_by_category = phenotype_category_df['HPO_Category'].value_counts().sort_values(ascending=True)

fig1, ax1 = plt.subplots(figsize=(10, max(5, 0.4 * len(phenotypes_by_category))))
ax1.barh(phenotypes_by_category.index.astype(str), phenotypes_by_category.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Number of distinct phenotypes', fontsize=12)
ax1.set_ylabel('HPO parent category', fontsize=12)
ax1.set_title('Distribution of phenotypes by parent HPO ontology category', fontsize=15, pad=15)
ax1.set_xlim(left=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(phenotypes_by_category.values):
    ax1.text(v + phenotypes_by_category.max() * 0.01, i, str(v), va='center', ha='left', fontsize=10)

fig1.text(0.5, -0.03, "Categories determined using HPO (Human Phenotype Ontology) codes: each phenotype's parent category "
                       "is its top-level ancestor under HP:0000118.",
          ha='center', va='top', fontsize=9, style='italic', color='#555555')

plt.tight_layout()
plt.show()

display(phenotypes_by_category.sort_values(ascending=False))

In [0]:
# ============================================================
# Chart 2: Phenotypes scored by Manual/each model, by parent HPO ontology category
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

df = combined_df_full_info.copy()

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# Re-attach HPO_Category using the mappings built earlier (phenotype_to_hpo, hpo_to_category
# must still be in memory from the HPO offline-parsing step)
df['HPO_ID'] = df['Phenotype'].map(phenotype_to_hpo)
df['HPO_Category'] = df['HPO_ID'].map(hpo_to_category)
df['HPO_Category'] = df['HPO_Category'].fillna('Unmapped / Other')

scored_by_category = df.groupby('HPO_Category')[score_cols].sum()

# Order categories the same way as Chart 1 (by total phenotype count) for consistency
scored_by_category = scored_by_category.reindex(phenotypes_by_category.index).dropna(how='all')

y = np.arange(len(scored_by_category))
height = 0.2

fig2, ax2 = plt.subplots(figsize=(11, max(6, 0.45 * len(scored_by_category))))
for i, col in enumerate(score_cols):
    ax2.barh(y + (i - 1.5) * height, scored_by_category[col], height,
             label=labels[col], color=colors[col])

ax2.set_yticks(y)
ax2.set_yticklabels(scored_by_category.index.astype(str))
ax2.set_xlabel('Number of subjects scored positive', fontsize=12)
ax2.set_ylabel('HPO parent category', fontsize=12)
ax2.set_title('Phenotypes scored by Manual and each model, by parent HPO ontology category', fontsize=15, pad=15)
ax2.set_xlim(left=0)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=4, frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig2.text(0.5, -0.02, "Categories determined using HPO (Human Phenotype Ontology) codes: each phenotype's parent category "
                       "is its top-level ancestor under HP:0000118.",
          ha='center', va='top', fontsize=9, style='italic', color='#555555')

plt.tight_layout()
plt.show()

display(scored_by_category)

In [0]:
def get_all_top_level_categories(hpo_id, parent_map, top_level_ids):
    visited = set()
    found = set()
    frontier = [hpo_id]
    while frontier:
        next_frontier = []
        for node in frontier:
            if node in visited:
                continue
            visited.add(node)
            if node in top_level_ids:
                found.add(node)
                continue  # don't traverse past a top-level hit
            for parent in parent_map.get(node, []):
                next_frontier.append(parent)
        frontier = next_frontier
    return found

multi_parent_report = {
    hpo_id: get_all_top_level_categories(hpo_id, parent_map, top_level_ids)
    for hpo_id in set(phenotype_to_hpo.values())
}
multi_parent_terms = {k: v for k, v in multi_parent_report.items() if len(v) > 1}
print(f"Phenotypes with more than one top-level HPO category: {len(multi_parent_terms)}")
for hpo_id, cats in multi_parent_terms.items():
    print(f"  {hpo_id}: {[id_to_label.get(c, c) for c in cats]}")

In [0]:
# ============================================================
# Check ontology depth (distance from HP:0000118) for every phenotype
# ============================================================

def compute_depth(hpo_id, parent_map, root='HP:0000118'):
    """Shortest number of is_a steps from hpo_id up to the root (HP:0000118).
    Returns None if no path to root is found."""
    if hpo_id == root:
        return 0
    from collections import deque
    visited = {hpo_id}
    queue = deque([(hpo_id, 0)])
    while queue:
        node, dist = queue.popleft()
        for parent in parent_map.get(node, []):
            if parent == root:
                return dist + 1
            if parent not in visited:
                visited.add(parent)
                queue.append((parent, dist + 1))
    return None  # no path found -- possible data issue, worth flagging

depth_records = []
for pheno, hpo_id in phenotype_to_hpo.items():
    depth = compute_depth(hpo_id, parent_map)
    depth_records.append({
        'Phenotype': pheno,
        'HPO_ID': hpo_id,
        'HPO_Label': id_to_label.get(hpo_id, ''),
        'Depth': depth,
    })

depth_df = pd.DataFrame(depth_records).sort_values('Depth')

print(f"Total phenotypes checked: {len(depth_df)}")
print(f"Unique depth values found: {sorted(depth_df['Depth'].dropna().unique())}")
print(f"\nDepth distribution:")
print(depth_df['Depth'].value_counts().sort_index())

no_path = depth_df[depth_df['Depth'].isna()]
if len(no_path) > 0:
    print(f"\nWARNING: {len(no_path)} phenotype(s) had no traceable path to HP:0000118 -- check these IDs:")
    display(no_path)

display(depth_df)

In [0]:
# Visualize the depth distribution
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 5))
depth_counts = depth_df['Depth'].value_counts().sort_index()
ax.bar(depth_counts.index.astype(str), depth_counts.values, color='#4C9BE8', edgecolor='white')
ax.set_xlabel('Ontology depth (steps from HP:0000118)', fontsize=12)
ax.set_ylabel('Number of phenotypes', fontsize=12)
ax.set_title('Distribution of phenotype specificity (ontology depth)', fontsize=15, pad=15)
ax.set_ylim(bottom=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, v in enumerate(depth_counts.values):
    ax.text(i, v + depth_counts.max() * 0.01, str(v), ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [0]:
def get_ancestor_at_depth(hpo_id, parent_map, target_depth, root='HP:0000118'):
    """Find the ancestor of hpo_id that sits at target_depth steps from root,
    along the shortest path. Returns None if hpo_id's depth is already <= target_depth
    with no ancestor exactly at that depth."""
    from collections import deque
    depth = compute_depth(hpo_id, parent_map, root)
    if depth is None or depth <= target_depth:
        return hpo_id  # already at or above the target depth -- keep as-is
    # BFS tracking path length, return first ancestor hit at target_depth
    visited = {hpo_id}
    queue = deque([(hpo_id, depth)])
    while queue:
        node, d = queue.popleft()
        if d == target_depth:
            return node
        for parent in parent_map.get(node, []):
            if parent not in visited:
                visited.add(parent)
                parent_depth = compute_depth(parent, parent_map, root)
                if parent_depth is not None:
                    queue.append((parent, parent_depth))
    return None

depth_df['Depth2_Ancestor'] = depth_df['HPO_ID'].apply(
    lambda h: get_ancestor_at_depth(h, parent_map, target_depth=2)
)
depth_df['Depth2_Label'] = depth_df['Depth2_Ancestor'].map(id_to_label)

n_original = depth_df['Phenotype'].nunique()
n_depth2_buckets = depth_df['Depth2_Ancestor'].nunique()
n_top_level = len(top_level_ids)

print(f"Original distinct phenotypes: {n_original}")
print(f"Distinct depth-2 buckets: {n_depth2_buckets}")
print(f"Distinct top-level (depth-1) categories: {n_top_level}")
print(f"\nHow many original phenotypes collapse into each depth-2 bucket:")
display(depth_df.groupby('Depth2_Label')['Phenotype'].apply(list))

In [0]:
# ============================================================
# Chart 1: Distribution of phenotypes by depth-2 HPO ontology bucket
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Build phenotype -> depth-2 ancestor mapping (reuses depth_df from the previous step)
phenotype_depth2_df = depth_df[['Phenotype', 'HPO_ID', 'Depth2_Ancestor', 'Depth2_Label']].copy()

phenotypes_by_depth2 = phenotype_depth2_df['Depth2_Label'].value_counts().sort_values(ascending=True)

fig1, ax1 = plt.subplots(figsize=(10, max(6, 0.3 * len(phenotypes_by_depth2))))
ax1.barh(phenotypes_by_depth2.index.astype(str), phenotypes_by_depth2.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Number of distinct phenotypes', fontsize=12)
ax1.set_ylabel('Depth-2 HPO ontology bucket', fontsize=12)
ax1.set_title('Distribution of phenotypes by depth-2 HPO ontology category', fontsize=15, pad=15)
ax1.set_xlim(left=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(phenotypes_by_depth2.values):
    ax1.text(v + phenotypes_by_depth2.max() * 0.01, i, str(v), va='center', ha='left', fontsize=9)

fig1.text(0.5, -0.02, "Categories determined using HPO (Human Phenotype Ontology) codes: each phenotype was rolled up "
                       "to its ancestor at depth 2 from HP:0000118, so all phenotypes are compared at the same ontology level.",
          ha='center', va='top', fontsize=9, style='italic', color='#555555')

plt.tight_layout()
plt.show()

display(phenotypes_by_depth2.sort_values(ascending=False))

In [0]:
# ============================================================
# Chart 2: Phenotypes scored by Manual/each model, by depth-2 HPO ontology bucket
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

df = combined_df_full_info.copy()

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# Map each row's Phenotype -> depth-2 ancestor label, using the lookup built earlier
phenotype_to_depth2_label = dict(zip(phenotype_depth2_df['Phenotype'], phenotype_depth2_df['Depth2_Label']))
df['Depth2_Category'] = df['Phenotype'].map(phenotype_to_depth2_label)
df['Depth2_Category'] = df['Depth2_Category'].fillna('Unmapped / Other')

scored_by_depth2 = df.groupby('Depth2_Category')[score_cols].sum()

# Order to match Chart 1 for consistency
scored_by_depth2 = scored_by_depth2.reindex(phenotypes_by_depth2.index).dropna(how='all')

y = np.arange(len(scored_by_depth2))
height = 0.2

fig2, ax2 = plt.subplots(figsize=(11, max(8, 0.35 * len(scored_by_depth2))))
for i, col in enumerate(score_cols):
    ax2.barh(y + (i - 1.5) * height, scored_by_depth2[col], height,
             label=labels[col], color=colors[col])

ax2.set_yticks(y)
ax2.set_yticklabels(scored_by_depth2.index.astype(str))
ax2.set_xlabel('Number of subjects scored positive', fontsize=12)
ax2.set_ylabel('Depth-2 HPO ontology bucket', fontsize=12)
ax2.set_title('Phenotypes scored by Manual and each model, by depth-2 HPO ontology category', fontsize=15, pad=15)
ax2.set_xlim(left=0)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.04), ncol=4, frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig2.text(0.5, -0.01, "Categories determined using HPO (Human Phenotype Ontology) codes: each phenotype was rolled up "
                       "to its ancestor at depth 2 from HP:0000118, so all phenotypes are compared at the same ontology level.",
          ha='center', va='top', fontsize=9, style='italic', color='#555555')

plt.tight_layout()
plt.show()

display(scored_by_depth2)

In [0]:
seven_imputed_ids = [7 WES IDs]
set(seven_imputed_ids) & set(results_GPT_full_df['Subject ID'])

In [0]:
blah blah blah stop the run right now

# Adjudication Splitting

In [0]:
# Randomly splitting the data into adjudication groups

import numpy as np

# Get per-subject disagreement stats
subject_stats = combined_df.groupby('Subject ID').agg(
    total_phenotypes=('Manual_Score', 'sum'),
    any_disagree=('any_disagree', 'sum')
).reset_index()

print(f"Total subjects: {len(subject_stats)}")
print(f"Mean phenotypes per subject: {subject_stats['total_phenotypes'].mean():.2f}")
print(f"Std phenotypes per subject: {subject_stats['total_phenotypes'].std():.2f}")

# Shuffle randomly
subject_stats = subject_stats.sample(frac=1, random_state=42).reset_index(drop=True)

# Split into 4 groups: 1 group of 50, 3 equal groups of remaining
n_total = len(subject_stats)
n_remaining = n_total - 50
n_per_group = n_remaining // 3
n_extra = n_remaining % 3  # distribute any remainder

print(f"\nGroup sizes: 50 + {n_per_group} + {n_per_group} + {n_per_group + n_extra}")

# Sort by phenotype count for stratified-style assignment
# Assign groups by cycling through sorted subjects so each group
# gets an even spread of high and low phenotype counts
subject_stats_sorted = subject_stats.sort_values('total_phenotypes').reset_index(drop=True)

# Assign group labels cycling 0-3 across sorted subjects
subject_stats_sorted['group'] = subject_stats_sorted.index % 4

# Remap group 0 to the "50 note" group and cap it at 50,
# redistributing overflow into groups 1-3
group0 = subject_stats_sorted[subject_stats_sorted['group'] == 0].copy()
overflow = group0.iloc[50:]
group0 = group0.iloc[:50]

# Redistribute overflow evenly into groups 1-3
for i, idx in enumerate(overflow.index):
    subject_stats_sorted.loc[idx, 'group'] = (i % 3) + 1

subject_stats_sorted.loc[group0.index, 'group'] = 0

# Report group stats
print("\n--- Group summary ---")
for g in range(4):
    grp = subject_stats_sorted[subject_stats_sorted['group'] == g]
    label = "Group 0 (n=50)" if g == 0 else f"Group {g}"
    print(f"{label}: n={len(grp)}, mean phenotypes={grp['total_phenotypes'].mean():.2f}, std={grp['total_phenotypes'].std():.2f}")

# Save group assignments
group_assignments = subject_stats_sorted[['Subject ID', 'group']]
display(group_assignments)

In [0]:
print("\n--- Group summary ---")
for g in range(4):
    grp = subject_stats_sorted[subject_stats_sorted['group'] == g]
    label = "Group 0 (n=50)" if g == 0 else f"Group {g}"
    n_notes = len(grp)
    total_pheno = grp['total_phenotypes'].sum()
    avg_pheno_per_note = grp['total_phenotypes'].mean()
    print(f"{label}:")
    print(f"  Notes:                    {n_notes}")
    print(f"  Total phenotypes:         {total_pheno}")
    print(f"  Avg phenotypes per note:  {avg_pheno_per_note:.2f}")
    print(f"  Std phenotypes per note:  {grp['total_phenotypes'].std():.2f}")
    print()

In [0]:
# Recompute subject_stats using disagreements 
subject_stats = combined_df.groupby('Subject ID').agg(
    total_disagreements=('any_disagree', 'sum')
).reset_index()

print(f"Total subjects: {len(subject_stats)}")
print(f"Total disagreements: {subject_stats['total_disagreements'].sum()}")
print(f"Mean disagreements per subject: {subject_stats['total_disagreements'].mean():.2f}")

results = []

for seed in range(1000):
    shuffled = subject_stats.sample(frac=1, random_state=seed).reset_index(drop=True)
    g0 = shuffled.iloc[:50]
    remaining = shuffled.iloc[50:].reset_index(drop=True)
    n_per_group = len(remaining) // 3
    n_extra = len(remaining) % 3
    group_sizes = [n_per_group + (1 if i < n_extra else 0) for i in range(3)]

    start = 0
    means = [g0['total_disagreements'].mean()]
    for size in group_sizes:
        grp = remaining.iloc[start:start+size]
        means.append(grp['total_disagreements'].mean())
        start += size

    results.append({'seed': seed, 'std_of_means': np.std(means), 'means': means})

results_df = pd.DataFrame(results).sort_values('std_of_means')
print("\nTop 10 most balanced seeds:")
display(results_df.head(10))

best_seed = results_df.iloc[0]['seed']
print(f"\nBest seed: {best_seed}")
print(f"Group means: {results_df.iloc[0]['means']}")

In [0]:
# Build the adjudication dataframes using the shuffled subject stats

subject_stats_shuffled = subject_stats.sample(frac=1, random_state=best_seed).reset_index(drop=True)

group0 = subject_stats_shuffled.iloc[:50].copy()
group0['group'] = 0

remaining = subject_stats_shuffled.iloc[50:].copy().reset_index(drop=True)
n_per_group = len(remaining) // 3
n_extra = len(remaining) % 3

group_sizes = [n_per_group + (1 if i < n_extra else 0) for i in range(3)]
print(f"Group sizes: {group_sizes}")

remaining['group'] = -1
start = 0
for i, size in enumerate(group_sizes):
    remaining.loc[start:start+size-1, 'group'] = i + 1
    start += size

subject_stats_sorted = pd.concat([group0, remaining], ignore_index=True)

print("--- Group summary ---")
for g in range(4):
    grp = subject_stats_sorted[subject_stats_sorted['group'] == g]
    label = "Group 0 (n=50)" if g == 0 else f"Group {g}"
    print(f"{label}:")
    print(f"  Notes:                      {len(grp)}")
    print(f"  Total disagreements:        {grp['total_disagreements'].sum()}")
    print(f"  Avg disagreements per note: {grp['total_disagreements'].mean():.2f}")
    print(f"  Std disagreements per note: {grp['total_disagreements'].std():.2f}")
    print()

print(f"Sum of all disagreements across groups: {subject_stats_sorted['total_disagreements'].sum()}")

group_assignments = subject_stats_sorted[['Subject ID', 'group']]
display(group_assignments)

In [0]:
print("--- subject_stats_sorted ---")
print(f"Total notes:         {subject_stats_sorted['Subject ID'].nunique()}")
print(f"Total disagreements: {subject_stats_sorted['total_disagreements'].sum()}")

print("\n--- combined_df ---")
print(f"Total notes:         {combined_df['Subject ID'].nunique()}")
print(f"Total disagreements: {combined_df['any_disagree'].sum()}")
print(f"Total rows:          {len(combined_df)}")

In [0]:
# Filter to only rows where any model disagrees with manual
adjudication_df = combined_df[combined_df['any_disagree'] == 1].copy()

# Join note text
adjudication_df = adjudication_df.merge(
    notes_enrolled4[['Subject ID', 'NOTE_TEXT']],
    on='Subject ID',
    how='left'
)

# Join group assignments
adjudication_df = adjudication_df.merge(
    group_assignments,
    on='Subject ID',
    how='left'
)

# Select and reorder columns
adjudication_df = adjudication_df[[
    'Subject ID',
    'Phenotype',
    'GPT_Score',
    'Justification',
    'Quote',
    'NOTE_TEXT',
    'Manual_Score',
    'GPT_Mini_Score',
    'Claude_Sonnet_Score',
    'group'
]]

print(f"Total rows: {len(adjudication_df)}")
print(f"Total subjects: {adjudication_df['Subject ID'].nunique()}")
print(f"Null NOTE_TEXT: {adjudication_df['NOTE_TEXT'].isna().sum()}")
print(f"Null group: {adjudication_df['group'].isna().sum()}")
display(adjudication_df.head())

In [0]:
# Checking that all dropped subjects are genuinely those where there are no disagreements on any of the phenotypes

no_disagree_subjects = combined_df.groupby('Subject ID')['any_disagree'].sum()
no_disagree_subjects = no_disagree_subjects[no_disagree_subjects == 0].index.tolist()
print(f"Subjects with no disagreements: {len(no_disagree_subjects)}")
print(f"Subject IDs: {no_disagree_subjects}")
print(f"Combined_df subjects: {combined_df['Subject ID'].nunique()}")
print(f"Adjudication_df subjects: {adjudication_df['Subject ID'].nunique()}")
print(f"Difference: {combined_df['Subject ID'].nunique() - adjudication_df['Subject ID'].nunique()}")

In [0]:
# Displaying the disagreements split as a table

summary_rows = []
for g in range(4):
    grp = adjudication_df[adjudication_df['group'] == g]
    label = "Group 0 (n=50)" if g == 0 else f"Group {g}"
    per_subject = grp.groupby('Subject ID').size()
    summary_rows.append({
        'Group': label,
        'Notes': grp['Subject ID'].nunique(),
        'Total Disagreements': len(grp),
        'Avg Disagreements per Note': round(per_subject.mean(), 2),
        'Std Disagreements per Note': round(per_subject.std(), 2)
    })

summary_table = pd.DataFrame(summary_rows)
print(summary_table.to_string(index=False))

In [0]:
print(adjudication_df.columns.tolist())

In [0]:
# Rebuild adjudication_df cleanly from combined_df
adjudication_df = combined_df[combined_df['any_disagree'] == 1].copy()

print(f"Rows with any disagreement: {len(adjudication_df)}")

# Join note text
adjudication_df = adjudication_df.merge(
    notes_enrolled4[['Subject ID', 'NOTE_TEXT']],
    on='Subject ID',
    how='left'
)
print(f"After NOTE_TEXT join: {len(adjudication_df)} rows, {adjudication_df['NOTE_TEXT'].isna().sum()} null NOTE_TEXT")

# Join group assignments
adjudication_df = adjudication_df.merge(
    group_assignments,
    on='Subject ID',
    how='left'
)
print(f"After group join: {len(adjudication_df)} rows, {adjudication_df['group'].isna().sum()} null group")

# Select and reorder columns
adjudication_df = adjudication_df[[
    'Subject ID',
    'Phenotype',
    'GPT_Score',
    'Justification',
    'Quote',
    'NOTE_TEXT',
    'Manual_Score',
    'GPT_Mini_Score',
    'Claude_Sonnet_Score',
    'group'
]]

# Split into four groups
group0_df = adjudication_df[adjudication_df['group'] == 0].copy()
group1_df = adjudication_df[adjudication_df['group'] == 1].copy()
group2_df = adjudication_df[adjudication_df['group'] == 2].copy()
group3_df = adjudication_df[adjudication_df['group'] == 3].copy()

print(f"\nGroup 0: {group0_df['Subject ID'].nunique()} subjects, {len(group0_df)} rows")
print(f"Group 1: {group1_df['Subject ID'].nunique()} subjects, {len(group1_df)} rows")
print(f"Group 2: {group2_df['Subject ID'].nunique()} subjects, {len(group2_df)} rows")
print(f"Group 3: {group3_df['Subject ID'].nunique()} subjects, {len(group3_df)} rows")
print(f"Total rows: {len(group0_df) + len(group1_df) + len(group2_df) + len(group3_df)}")
display(adjudication_df.head())

In [0]:
display(group0_df)
print(len(group0_df))
display(group1_df)
print(len(group1_df))
display(group2_df)
print(len(group2_df))
display(group3_df)
print(len(group3_df))

In [0]:
# Dropping unnecessary columns
cols = ['Subject ID', 'Phenotype', 'Justification', 'NOTE_TEXT', 'Quote']

group0_df = group0_df[cols].copy()
group1_df = group1_df[cols].copy()
group2_df = group2_df[cols].copy()
group3_df = group3_df[cols].copy()

display(group0_df.head())

In [0]:
group0_df.to_csv("/Workspace/Users/user/user/Kappa_Adjudications_Unscored.csv", index=False)
group1_df.to_csv("/Workspace/Users/user/user/Burak_Adjudications_Unscored.csv", index=False)
group2_df.to_csv("/Workspace/Users/user/user/Angie_Adjudications_Unscored.csv", index=False)
group3_df.to_csv("/Workspace/Users/user/user/Dustin_Adjudications_Unscored.csv", index=False)
print("Saved all four CSVs")

In [0]:
for name, df in [('Adjudication', group0_df), ('Burak', group1_df), ('Angie', group2_df), ('Dustin', group3_df)]:
    print(f"{name}:")
    print(f"  Rows: {len(df)}")
    print(f"  Unique subjects: {df['Subject ID'].nunique()}")
    print(f"  Phenotypes needing scoring: {len(df)}")
    print(f"  Estimated tokens (chars/4): {df['NOTE_TEXT'].str.len().sum() // 4:,}")
    print("---")

# Fixing Mistake

In [0]:
checklist_terms_now = checklist_filtered[['Term from checklist']].drop_duplicates()

pipeline_join = manual_df.merge(
    checklist_terms_now,
    left_on='Phenotype',
    right_on='Term from checklist',
    how='inner'
)

in_manual_score = set(zip(manual_score['Subject ID'], manual_score['Phenotype']))
pipeline_join['pair'] = list(zip(pipeline_join['Subject ID'], pipeline_join['Phenotype']))

pipeline_missing = pipeline_join[~pipeline_join['pair'].isin(in_manual_score)]

print(f"Rows in manual_df x current checklist_filtered: {len(pipeline_join)}")
print(f"Rows currently in manual_score: {len(manual_score)}")
print(f"Present per manual_df, absent from manual_score: {len(pipeline_missing)}")
print()
print(pipeline_missing.groupby('Phenotype').size().sort_values(ascending=False))

In [0]:
cohort_ids = set(combined_df['Subject ID'])

pipeline_missing_cohort = pipeline_missing[pipeline_missing['Subject ID'].isin(cohort_ids)]

print(f"Present per manual_df, absent from manual_score (unfiltered): {len(pipeline_missing)}")
print(f"Same, restricted to subjects in combined_df: {len(pipeline_missing_cohort)}")
print()
print(pipeline_missing_cohort.groupby('Phenotype').size().sort_values(ascending=False))

In [0]:
group0 = pd.read_csv("/Workspace/Users/user/user/Kappa_Adjudications_Unscored.csv")
group1 = pd.read_csv("/Workspace/Users/user/user/Burak_Adjudications_Unscored.csv")
group2 = pd.read_csv("/Workspace/Users/user/user/Angie_Adjudications_Unscored.csv")
group3 = pd.read_csv("/Workspace/Users/user/user/Dustin_Adjudications_Unscored.csv")

all_adjudications = pd.concat([group0, group1, group2, group3], ignore_index=True)

print(f"Rows loaded per file: {len(group0)}, {len(group1)}, {len(group2)}, {len(group3)}")
print(f"Total concatenated: {len(all_adjudications)}")
print(f"Columns: {list(all_adjudications.columns)}")
print()

adjudicated_pairs = set(zip(all_adjudications['Subject ID'], all_adjudications['Phenotype']))

missing_pairs = pd.Series(
    list(zip(pipeline_missing_cohort['Subject ID'], pipeline_missing_cohort['Phenotype'])),
    index=pipeline_missing_cohort.index
)
needs_adjudication = pipeline_missing_cohort[~missing_pairs.isin(adjudicated_pairs)]

print(f"Cells checked: {len(pipeline_missing_cohort)}")
print(f"Already present in adjudication files: {len(pipeline_missing_cohort) - len(needs_adjudication)}")
print(f"NOT in adjudication files -- needs new adjudication: {len(needs_adjudication)}")
print()
print(needs_adjudication.groupby('Phenotype').size().sort_values(ascending=False))
print()
print(needs_adjudication[['Subject ID', 'Phenotype']].sort_values(['Phenotype', 'Subject ID']).to_string(index=False))

In [0]:
check_full = pipeline_missing_cohort.copy()
check_full['pair'] = list(zip(check_full['Subject ID'], check_full['Phenotype']))
check_full['already_adjudicated'] = check_full['pair'].isin(adjudicated_pairs)

summary = check_full.groupby('Phenotype').agg(
    missing_from_manual_score=('pair', 'count'),
    already_adjudicated=('already_adjudicated', 'sum')
)
summary['still_needs_adjudication'] = summary['missing_from_manual_score'] - summary['already_adjudicated']

print(summary)
print()
print(f"Column totals: {summary['missing_from_manual_score'].sum()}, {summary['already_adjudicated'].sum()}, {summary['still_needs_adjudication'].sum()}")

In [0]:
final_needs_adjudication = needs_adjudication.merge(
    combined_df[['Subject ID', 'Phenotype', 'Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']],
    on=['Subject ID', 'Phenotype'],
    how='left'
).merge(
    notes_enrolled4[['Subject ID', 'NOTE_TEXT']],
    on='Subject ID',
    how='left'
)

print(f"Rows: {len(final_needs_adjudication)}")
print(f"Rows missing note text (join problem if > 0): {final_needs_adjudication['NOTE_TEXT'].isna().sum()}")
print()
print(final_needs_adjudication[['Subject ID', 'Phenotype', 'Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']].sort_values(['Phenotype', 'Subject ID']).to_string(index=False))

In [0]:
final_needs_adjudication = final_needs_adjudication.merge(
    results_GPT_full_df[['Subject ID', 'Phenotype', 'Justification', 'Quote']],
    on=['Subject ID', 'Phenotype'],
    how='left'
)

print(f"Rows: {len(final_needs_adjudication)}")
print(f"Rows missing Justification after merge (join problem if > 0): {final_needs_adjudication['Justification'].isna().sum()}")
print()

display(final_needs_adjudication)

In [0]:
print(list(final_needs_adjudication.columns))

In [0]:
current_terms = set(checklist_filtered['Term from checklist'].unique())

pairs_to_check = [
    ('Autism Spectrum Disorder (ASD)', 'Autism Spectrum Disorder (ASD'),
    ('Diabetes Mellitus Type I (insulin dependent)', 'Diabetes Mellitus Type I (insulin dependent'),
    ('Diabetes Mellitus Type II (non-insulin dependent)', 'Diabetes Mellitus Type II (non-insulin dependent'),
    ('Double Outlet Right Ventricle (DORV)', 'Double Outlet Right Ventricle (DORV'),
    ('Hypertrophic cardiomyopathy (HCM)', 'Hypertrophic cardiomyopathy (HCM'),
    ('Tetralogy of Fallot (TOF)', 'Tetralogy of Fallot (TOF'),
]

print(f"Unique terms in checklist_filtered right now: {len(current_terms)}")
print()
for closed, unclosed in pairs_to_check:
    print(f"closed form   {'PRESENT' if closed in current_terms else 'absent  '}  {closed!r}")
    print(f"unclosed form {'PRESENT' if unclosed in current_terms else 'absent  '}  {unclosed!r}")
    print()

In [0]:
dropped_355 = pd.read_csv("/Workspace/Users/user/user/manual_score_dropped_positives.csv")

cohort_ids = set(combined_df['Subject ID'])
dropped_cohort = dropped_355[dropped_355['Subject ID'].isin(cohort_ids)]

pairs = pd.Series(
    list(zip(dropped_cohort['Subject ID'], dropped_cohort['Phenotype'])),
    index=dropped_cohort.index
)
not_in_adjudication = dropped_cohort[~pairs.isin(adjudicated_pairs)]

print(f"Raw-source-confirmed cells, restricted to cohort: {len(dropped_cohort)}")
print(f"Not present in adjudication files: {len(not_in_adjudication)}")
print()
print(not_in_adjudication.groupby('Phenotype').size().sort_values(ascending=False))

In [0]:
to_append = not_in_adjudication.merge(
    combined_df[['Subject ID', 'Phenotype', 'Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']],
    on=['Subject ID', 'Phenotype'], how='left'
).merge(
    notes_enrolled4[['Subject ID', 'NOTE_TEXT']],
    on='Subject ID', how='left'
).merge(
    results_GPT_full_df[['Subject ID', 'Phenotype', 'Justification', 'Quote']],
    on=['Subject ID', 'Phenotype'], how='left'
)

print(f"Rows: {len(to_append)}")
print(f"Missing NOTE_TEXT: {to_append['NOTE_TEXT'].isna().sum()}")
print(f"Missing Justification: {to_append['Justification'].isna().sum()}")
print()
cols_show = ['Subject ID', 'Phenotype', 'Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
print(to_append[cols_show].sort_values(['Phenotype', 'Subject ID']).to_string(index=False))

In [0]:
group3_fresh = pd.read_csv("/Workspace/Users/user/user/Dustin_Adjudications_Unscored.csv")
dustin_schema = list(group3_fresh.columns)

dustin_new_rows = to_append[dustin_schema].copy()

dustin_combined = pd.concat([group3_fresh, dustin_new_rows], ignore_index=True)

dup_check = dustin_combined.duplicated(subset=['Subject ID', 'Phenotype']).sum()

print(f"Dustin's original rows: {len(group3_fresh)}")
print(f"New rows being added: {len(dustin_new_rows)}")
print(f"Combined total: {len(dustin_combined)}")
print(f"Duplicate Subject ID x Phenotype pairs after combining: {dup_check}")
print(f"Schema: {dustin_schema}")

out_path = "/Workspace/Users/user/user/Dustin_Adjudications_Unscored_updated.csv"
dustin_combined.to_csv(out_path, index=False)
print()
print(f"Saved to {out_path}")